# CMPE 260 Project 2 — Final notebook: REINFORCE and A2C experiments, and comparison with DQN

This is the **final notebook** of Project 2. It contains every experiment of the project in one place:

* the experiments **already finished** in the earlier notebooks (`CMPE260_Project2_PG_Pong.ipynb` and
  `CMPE260_Project2_Extended.ipynb`). They are marked **done (earlier)** and are **never retrained** here.
  Their saved results are read from `CMPE260_Project2/`;
* **4 new REINFORCE experiments (R1–R4)** and **8 new A2C experiments (A1–A8)**, which this notebook trains;
* the full **results tables**, one table per experiment, seed summaries and learning curves;
* the **final tuned hyper-parameters** of each algorithm, selected from all runs;
* the **final comparison of DQN (Project 1), REINFORCE and A2C**. Project 1 is **only read, never re-run**:
  its recorded results (training results, logs, evaluation scores and Q-value over-estimation) are stored
  in `CMPE260_Project1/` and are used only in this final comparison (Section 17).

> **Hardware note.** Project 1 (DQN) was trained on Google Colab (NVIDIA L4 GPU) with a
> single environment. Project 2 (REINFORCE, A2C) is trained on NVIDIA RTX GPUs with 16 parallel
> environments. Wall-clock times are therefore **not directly comparable** between the two projects.
> Frames, episodes, gradient updates, evaluation scores and value bias do not depend on the hardware and
> are the main basis of the comparison. Every table shows the GPU that produced each result.

| Section | Content |
|---|---|
| 1–11 | Setup, configuration, Atari preprocessing, networks, REINFORCE, A2C, trainer, correctness checks (same code as the earlier notebooks, plus the separate-critic option for A2C) |
| 12 | **Experiment plan**: every experiment, earlier and new, with its status |
| 13 | **Run the queue** of new experiments (skips everything that is already done) |
| 14 | Evaluation of the REINFORCE and A2C models (30 games) and critic bias |
| 15 | Results: all runs, one table per experiment (with GPU and variance columns), seed summaries, earlier sweeps |
| 16 | Learning curves of the experiments, A2C n-step bias-variance plot |
| 17 | **Final tuned hyper-parameters** and the **final comparison: DQN (Project 1 results) vs REINFORCE vs A2C** |
| 18 | How to read the results |

**Run status legend:** `done (earlier)` = finished before this notebook, never retrained ·
`done` = new run, finished · `partial` = new run, started and resumable · `to run` = new run, not started.

## 1. Setup and how to run this notebook

1. **Python 3.10–3.12** with Jupyter (`pip install jupyterlab ipykernel`).
2. **PyTorch with CUDA** (the default PyPI wheel on Windows is CPU-only):
   ```
   pip install torch --index-url https://download.pytorch.org/whl/cu128
   ```
3. Open this notebook **from the repository folder**. It reads and writes `./CMPE260_Project2/`
   (Project 2 results) and reads `./CMPE260_Project1/` (Project 1 DQN results, already in the repository).
4. **Run all cells.** The queue in Section 13 trains only the new experiments that are not finished.
   Every run checkpoints every 250k frames and resumes when the cell is re-run. To stop cleanly between
   runs, create an empty file `CMPE260_Project2/STOP` (delete it to continue).
5. **Choosing what to train.** `RUN_EXPERIMENTS` in Section 12 switches each new experiment on or off,
   so the work can be split between machines (for example R1–R4 on one PC and A1–A8 on another).
   After training, copy each new run folder (`results.json`, `episodes.npz`, `tensorboard/`) and its
   `eval_*.json` into `CMPE260_Project2/` of the shared repository, then re-run Sections 14–17.

**Without an NVIDIA GPU** (for example on a laptop) the notebook runs in *analysis-only* mode: training
and new evaluations are skipped, and every table and plot is rebuilt from the saved results.

**Troubleshooting on Windows**
* `OSError: [WinError 1114] ... c10.dll` when importing torch means an old `pyzmq` (≤ 25) is loaded.
  Fix: `pip install --upgrade "pyzmq>=26"` (the install cell does this), then restart the kernel.
* Environment workers start with `spawn` on Windows, which is why the environment code is written to
  `pong_envs.py` (Section 4). Each worker uses one CPU core; 16 workers need a CPU with 16+ threads for
  full speed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
%pip install -q "gymnasium==1.3.0" "ale-py==0.12.1" "opencv-python-headless>=4.12" "tensorboard>=2.15" "pyzmq>=26" pandas matplotlib tabulate psutil

In [ ]:
import collections
import contextlib
import functools
import importlib
import json
import math
import os
import platform
import random
import shutil
import sys
import time
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import ale_py
import cv2
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import display
from torch.utils.tensorboard import SummaryWriter

torch.distributions.Distribution.set_default_validate_args(False)  # skip per-step arg checks
gym.register_envs(ale_py)
cv2.setNumThreads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Training needs an NVIDIA GPU. Without one the notebook runs in analysis-only mode: every table and plot
# is rebuilt from the saved results, and the training and evaluation cells skip their work.
TRAINING_AVAILABLE = device.type == "cuda"
if not TRAINING_AVAILABLE:
    print("No CUDA GPU found -> ANALYSIS-ONLY MODE (training and new evaluations are skipped).\n"
          "To train, install the CUDA build of PyTorch "
          "(pip install torch --index-url https://download.pytorch.org/whl/cu128) and restart the kernel.\n")

# GPU speed-ups that do not change the algorithms.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

GPU_NAME = torch.cuda.get_device_name(0) if TRAINING_AVAILABLE else "none (CPU only)"
GPU_CAPABILITY = torch.cuda.get_device_capability(0) if TRAINING_AVAILABLE else (0, 0)
# bf16 autocast on Ampere/Ada and newer (RTX 30xx/40xx, sm_80+). Older GPUs stay fp32/TF32.
AMP_DTYPE = torch.bfloat16 if GPU_CAPABILITY[0] >= 8 else None
CPU_COUNT = os.cpu_count() or 1
IS_WINDOWS = platform.system() == "Windows"

print("OS        :", platform.platform())
print("Python    :", platform.python_version())
print("Gymnasium :", gym.__version__)
print("ALE       :", ale_py.__version__)
print("OpenCV    :", cv2.__version__)
print("PyTorch   :", torch.__version__, "CUDA", torch.version.cuda)
if TRAINING_AVAILABLE:
    print("GPU       :", GPU_NAME, "capability", GPU_CAPABILITY,
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
else:
    print("GPU       :", GPU_NAME)
print("AMP dtype :", AMP_DTYPE)
print("CPU cores :", CPU_COUNT, f"(physical {psutil.cpu_count(logical=False)})")
print("RAM       :", f"{psutil.virtual_memory().total / 2**30:.1f} GiB")
if CPU_COUNT < 8:
    print("WARNING: fewer than 8 logical CPUs. Training works but env stepping will be the bottleneck.")

## 2. Folders

Project 1 results are **only read** from `P1_ROOT` (`./CMPE260_Project1/`: the recorded `results.json`,
evaluation files and TensorBoard logs of the DQN runs; no checkpoints), and only in Section 17. Project 2 runs are
**read and written** in `P2_ROOT` (`./CMPE260_Project2/`). You can override both with the environment
variables `CMPE260_P1_ROOT` and `CMPE260_P2_ROOT`.

In [ ]:
PROJECT_DIR = Path.cwd()
P1_ROOT = Path(os.environ.get("CMPE260_P1_ROOT", PROJECT_DIR / "CMPE260_Project1"))
P2_ROOT = Path(os.environ.get("CMPE260_P2_ROOT", PROJECT_DIR / "CMPE260_Project2"))
P2_ROOT.mkdir(parents=True, exist_ok=True)

print("Notebook folder    :", PROJECT_DIR)
print("Project 1 artifacts:", P1_ROOT, "(exists)" if P1_ROOT.exists() else
      "(NOT FOUND - the DQN rows of the comparison will be missing)")
if P1_ROOT.exists():
    print("  runs:", sorted(p.name for p in P1_ROOT.iterdir() if p.is_dir()))
print("Project 2 artifacts:", P2_ROOT)
free_gb = shutil.disk_usage(P2_ROOT).free / 2**30
print(f"Free disk at P2_ROOT: {free_gb:.1f} GiB (all runs together need roughly 1-2 GiB)")

## 3. Configuration

One dataclass holds both algorithms' settings, so the controlled differences are explicit.
The values below are the **starting points** of the first notebook. The tuned settings and every
experiment built on them are listed in Section 12.

| Setting | REINFORCE | A2C | Why |
|---|---|---|---|
| parallel envs | 16 | 16 | same data throughput for both PG methods |
| rollout length | 512 steps/env (8,192 states/update) | 5 steps/env (80 states/update) | MC needs complete returns; TD bootstraps after *n* steps |
| optimizer | Adam, lr 5e-4 | RMSprop(α=0.99, ε=1e-5), lr 7e-4 | standard choices from the A2C paper and its reference implementations |
| entropy bonus | 0.01 | 0.01 | stops the policy from collapsing too early |
| value loss coef | — | 0.5 | |
| grad-norm clip | 1.0 | 0.5 | |
| return processing | per-point discounted return, standardised per batch | n-step / GAE(λ) advantage | variance reduction |

Options added after the first notebook (their defaults leave every earlier run unchanged):

| Option | Used by | Meaning |
|---|---|---|
| `value_baseline` | REINFORCE | learned V(s) baseline with Monte-Carlo targets (no bootstrapping) |
| `lr_schedule` | both | `"constant"` or `"linear"` (anneal to 0 at `max_frames`, as in the A2C paper) |
| `shared_torso` | A2C | `True`: actor and critic share the CNN. `False`: the critic has its own CNN (experiment A8) |

In [ ]:
@dataclass(frozen=True)
class PGConfig:
    algorithm: str                     # "reinforce" or "a2c"
    env_id: str = "ALE/Pong-v5"
    seed: int = 42
    gamma: float = 0.99
    num_envs: int = 16
    rollout_steps: int = 512           # steps per env between updates
    optimizer: str = "adam"            # "adam" or "rmsprop"
    learning_rate: float = 5e-4
    adam_eps: float = 1e-5
    rmsprop_alpha: float = 0.99
    rmsprop_eps: float = 1e-5
    entropy_coef: float = 0.01
    value_coef: float = 0.5            # A2C only
    max_grad_norm: float = 0.5
    gae_lambda: float = 1.0            # A2C only; 1.0 = plain n-step returns
    point_reset: bool = True           # REINFORCE only; reset the return at every Pong point
    normalize_returns: bool = True     # REINFORCE only; standardise returns per batch
    update_chunk: int = 4096           # GPU mini-batch for REINFORCE gradient accumulation
    use_amp: bool = True               # bf16 autocast on Ampere+ GPUs
    max_frames: int = 10_000_000       # agent steps, summed over all envs
    reward_bound: float = 19.0         # identical to Project 1
    min_episodes_to_solve: int = 100   # identical to Project 1
    convergence_threshold: float = 15.0  # Project 1 comparison threshold
    checkpoint_every: int = 250_000
    diagnostic_every: int = 10_000
    best_save_interval: int = 50_000   # rate-limits best.pt writes
    print_every_seconds: float = 30.0
    value_baseline: bool = False       # REINFORCE only: learned V(s) baseline, Monte-Carlo targets
    lr_schedule: str = "constant"      # "constant" or "linear" (anneal to 0 at max_frames)
    shared_torso: bool = True          # A2C only: False gives the critic its own CNN torso (experiment A8)


REINFORCE_CONFIG = PGConfig(
    algorithm="reinforce",
    rollout_steps=512,
    optimizer="adam",
    learning_rate=5e-4,
    entropy_coef=0.01,
    value_coef=0.0,
    max_grad_norm=1.0,
    point_reset=True,
    normalize_returns=True,
    max_frames=15_000_000,
)

A2C_CONFIG = PGConfig(
    algorithm="a2c",
    rollout_steps=5,
    optimizer="rmsprop",
    learning_rate=7e-4,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    gae_lambda=1.0,
    point_reset=False,
    normalize_returns=False,
    max_frames=10_000_000,
)

ENV_ID, SEED, GAMMA = REINFORCE_CONFIG.env_id, REINFORCE_CONFIG.seed, REINFORCE_CONFIG.gamma
print(json.dumps({"REINFORCE": asdict(REINFORCE_CONFIG), "A2C": asdict(A2C_CONFIG)}, indent=2))

## 4. Atari preprocessing (identical to Project 1)

The wrapper classes are copied unchanged from Project 1. The next cell writes them to `pong_envs.py`
next to the notebook and imports them. A separate module is needed because Windows starts worker
processes with `spawn`, and spawned workers can only rebuild an environment from code they can
import; they cannot use classes defined inside a notebook.

`make_vec_env` runs `num_envs` copies, each in its own process. It uses `SAME_STEP` auto-reset:
when an episode ends, the reset observation comes back on the same step. This keeps rollout
bookkeeping simple and correct.

In [ ]:
PONG_ENVS_SOURCE = r"""
# Auto-generated by CMPE260_Project2_PG_Pong.ipynb. Identical preprocessing to Project 1.
import collections

import ale_py
import cv2
import gymnasium as gym
import numpy as np

gym.register_envs(ale_py)
cv2.setNumThreads(1)  # one OpenCV thread per env worker avoids CPU oversubscription


class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env, skip=4):
        super().__init__(env)
        self.skip = int(skip)
        self.frames = collections.deque(maxlen=2)

    def reset(self, **kwargs):
        self.frames.clear()
        obs, info = self.env.reset(**kwargs)
        self.frames.append(obs)
        return obs, info

    def step(self, action):
        total_reward = 0.0
        terminated = truncated = False
        info = {}
        for _ in range(self.skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            self.frames.append(obs)
            total_reward += float(reward)
            if terminated or truncated:
                break
        return np.maximum.reduce(tuple(self.frames)), total_reward, terminated, truncated, info


class FireResetEnv(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        meanings = env.unwrapped.get_action_meanings()
        if len(meanings) < 3 or meanings[1] != "FIRE":
            raise ValueError(f"Unexpected Pong actions: {meanings}")

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for action in (1, 2):
            obs, _, terminated, truncated, info = self.env.step(action)
            if terminated or truncated:
                obs, info = self.env.reset()
        return obs, info


class ProcessFrame84(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(1, 84, 84), dtype=np.uint8)

    def observation(self, frame):
        if frame.shape not in ((210, 160, 3), (250, 160, 3)):
            raise ValueError(f"Unexpected Atari frame shape: {frame.shape}")
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (84, 110), interpolation=cv2.INTER_AREA)
        return resized[18:102, :][None, ...].astype(np.uint8, copy=False)


class FrameStack4(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.frames = collections.deque(maxlen=4)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(4, 84, 84), dtype=np.uint8)

    def _stack(self):
        return np.concatenate(tuple(self.frames), axis=0)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        zero = np.zeros_like(obs)
        self.frames.clear()
        self.frames.extend([zero.copy(), zero.copy(), zero.copy(), obs.copy()])
        return self._stack(), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.frames.append(obs.copy())
        return self._stack(), reward, terminated, truncated, info


def make_env(env_id="ALE/Pong-v5"):
    env = gym.make(env_id, frameskip=1, repeat_action_probability=0.0, full_action_space=False)
    env = MaxAndSkipEnv(env, skip=4)
    env = FireResetEnv(env)
    env = ProcessFrame84(env)
    env = FrameStack4(env)
    return env
"""

_module_path = PROJECT_DIR / "pong_envs.py"
if not _module_path.exists() or _module_path.read_text(encoding="utf-8") != PONG_ENVS_SOURCE:
    _module_path.write_text(PONG_ENVS_SOURCE, encoding="utf-8")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import pong_envs
importlib.reload(pong_envs)
from pong_envs import FireResetEnv, FrameStack4, MaxAndSkipEnv, ProcessFrame84, make_env

VECTOR_BACKEND = "async"  # "sync" runs all envs in this process (slow; debugging only)
MP_CONTEXT = "spawn" if IS_WINDOWS else "fork"


def make_vec_env(num_envs, env_id=ENV_ID, backend=None):
    backend = backend or VECTOR_BACKEND
    fns = [functools.partial(make_env, env_id) for _ in range(num_envs)]  # picklable for spawn
    mode = gym.vector.AutoresetMode.SAME_STEP
    if backend == "async":
        return gym.vector.AsyncVectorEnv(fns, shared_memory=True, context=MP_CONTEXT, autoreset_mode=mode)
    return gym.vector.SyncVectorEnv(fns, autoreset_mode=mode)


print("Wrote", _module_path, "| worker start method:", MP_CONTEXT)

### Preflight test and throughput benchmark

Do not continue if an assertion fails. The benchmark shows the environment-only throughput
ceiling of this runtime, in agent steps per second.

In [ ]:
if not TRAINING_AVAILABLE:
    print("Analysis-only mode: preflight benchmark skipped.")
else:
    test_env = make_env()
    obs0, _ = test_env.reset(seed=SEED)
    obs1, *_ = test_env.step(test_env.action_space.sample())
    assert obs0.shape == (4, 84, 84) and obs0.dtype == np.uint8
    assert not np.shares_memory(obs0, obs1) and np.any(obs0 != obs1)
    print("Single env :", obs0.shape, obs0.dtype, test_env.unwrapped.get_action_meanings())
    test_env.close()

    bench = make_vec_env(REINFORCE_CONFIG.num_envs)
    obs, _ = bench.reset(seed=SEED)
    assert obs.shape == (REINFORCE_CONFIG.num_envs, 4, 84, 84)
    steps, t0, n_done = 0, time.perf_counter(), 0
    for _ in range(300):
        obs, r, term, trunc, info = bench.step(bench.action_space.sample())
        steps += REINFORCE_CONFIG.num_envs
        n_done += int(np.sum(term | trunc))
    bench_fps = steps / (time.perf_counter() - t0)
    bench.close()
    print(f"Vector env : {REINFORCE_CONFIG.num_envs} envs, {bench_fps:,.0f} agent steps/s (env only, random actions)")

## 5. Networks

All methods use the **same CNN torso** as the Project 1 DQN: conv 32-8/4 → 64-4/2 → 64-3/1 → FC 512.

* **REINFORCE** uses the torso plus a policy head with 6 logits.
* **A2C** uses the torso shared by a policy head and a value head *V(s)*.

Orthogonal initialisation follows the A2C reference implementations: gain √2 in the torso,
0.01 on the policy head (near-uniform initial policy) and 1.0 on the value head.
`DQN` and `NoisyDQN` are copied from Project 1 only so that their checkpoints can be loaded
for evaluation.

With `shared_torso=False` (experiment A8) the critic gets its own copy of the CNN torso and FC layer, so the
value loss no longer changes the features the policy uses. The default shared network is built exactly
as before, so every earlier checkpoint still loads.

In [ ]:
def atari_torso(in_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        nn.Flatten(),
    )


class PGNet(nn.Module):
    def __init__(self, input_shape, n_actions, with_value, shared_torso=True):
        super().__init__()
        self.features = atari_torso(input_shape[0])
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, *input_shape)).shape[1]
        self.fc = nn.Sequential(nn.Linear(feature_size, 512), nn.ReLU())
        self.policy_head = nn.Linear(512, n_actions)
        self.value_head = nn.Linear(512, 1) if with_value else None
        # Experiment A8: the critic gets its own torso. With shared_torso=True no module is added, so the
        # network (and its initialisation) is identical to the earlier notebooks.
        self.separate_critic = bool(with_value and not shared_torso)
        if self.separate_critic:
            self.value_features = atari_torso(input_shape[0])
            self.value_fc = nn.Sequential(nn.Linear(feature_size, 512), nn.ReLU())
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.orthogonal_(m.weight, gain=math.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.policy_head.weight, gain=0.01)
        if self.value_head is not None:
            nn.init.orthogonal_(self.value_head.weight, gain=1.0)

    def forward(self, x):
        x = x.float().div(255.0)
        h = self.fc(self.features(x))
        logits = self.policy_head(h)
        if self.value_head is None:
            return logits, None
        hv = self.value_fc(self.value_features(x)) if self.separate_critic else h
        return logits, self.value_head(hv).squeeze(-1)


def amp_context(enabled):
    if enabled and AMP_DTYPE is not None and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return contextlib.nullcontext()


def forward_fp32(net, obs, use_amp):
    # Run the CNN under autocast, but return fp32 logits/values for stable losses.
    with amp_context(use_amp):
        logits, value = net(obs)
    return logits.float(), (None if value is None else value.float())


def build_pg_net(config, obs_shape, n_actions):
    return PGNet(obs_shape, n_actions,
                 with_value=(config.algorithm == "a2c" or config.value_baseline),
                 shared_torso=config.shared_torso).to(device)


def build_optimizer(net, config):
    if config.optimizer == "adam":
        try:
            return optim.Adam(net.parameters(), lr=config.learning_rate, eps=config.adam_eps, fused=True)
        except (RuntimeError, TypeError):
            return optim.Adam(net.parameters(), lr=config.learning_rate, eps=config.adam_eps)
    if config.optimizer == "rmsprop":
        return optim.RMSprop(net.parameters(), lr=config.learning_rate,
                             alpha=config.rmsprop_alpha, eps=config.rmsprop_eps)
    raise ValueError(config.optimizer)


def count_params(net):
    return sum(p.numel() for p in net.parameters())


# ---- Project 1 networks (only for loading DQN checkpoints) ----
class DQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.features = atari_torso(input_shape[0])
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, *input_shape)).shape[1]
        self.head = nn.Sequential(nn.Linear(feature_size, 512), nn.ReLU(), nn.Linear(512, n_actions))

    def forward(self, x):
        return self.head(self.features(x.float().div(255.0)))


class NoisyLinear(nn.Module):
    def __init__(self, in_features, out_features, sigma0=0.5):
        super().__init__()
        self.in_features, self.out_features = int(in_features), int(out_features)
        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))
        self.register_buffer("weight_epsilon", torch.zeros(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))
        self.register_buffer("bias_epsilon", torch.zeros(out_features))
        nn.init.zeros_(self.weight_mu); nn.init.zeros_(self.bias_mu)
        nn.init.zeros_(self.weight_sigma); nn.init.zeros_(self.bias_sigma)

    def forward(self, x):
        if self.training:
            w = self.weight_mu + self.weight_sigma * self.weight_epsilon
            b = self.bias_mu + self.bias_sigma * self.bias_epsilon
        else:
            w, b = self.weight_mu, self.bias_mu
        return F.linear(x, w, b)


class NoisyDQN(nn.Module):
    def __init__(self, input_shape, n_actions, sigma0=0.5):
        super().__init__()
        self.features = atari_torso(input_shape[0])
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, *input_shape)).shape[1]
        self.fc1 = NoisyLinear(feature_size, 512, sigma0)
        self.fc2 = NoisyLinear(512, n_actions, sigma0)

    def forward(self, x):
        x = self.features(x.float().div(255.0))
        return self.fc2(torch.relu(self.fc1(x)))


_probe = PGNet((4, 84, 84), 6, with_value=True)
print("PGNet (A2C) parameters      :", f"{count_params(_probe):,}")
print("PGNet (REINFORCE) parameters:", f"{count_params(PGNet((4, 84, 84), 6, with_value=False)):,}")
print("DQN parameters              :", f"{count_params(DQN((4, 84, 84), 6)):,}")
del _probe
print("PGNet (A2C, separate critic) parameters:",
      f"{count_params(PGNet((4, 84, 84), 6, with_value=True, shared_torso=False)):,}")

## 6. Shared run bookkeeping

`RunMonitor` matches the Project 1 runner. It logs the same TensorBoard tags
(`episode/reward`, `episode/reward_100`, `system/fps`, `system/wall_minutes`, `train/*`), writes
`latest.pt`, `best.pt` and `results.json` atomically, and applies the Project 1 stopping rule.
It also saves `episodes.npz` with the frame, reward, length and wall-clock time of every episode.
The comparison uses this file because 16 parallel environments can finish episodes on the same
frame, and TensorBoard would merge those events.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _replace(tmp, path, attempts=20):
    # On Windows, antivirus/indexers can briefly lock a file; retry the atomic rename.
    for i in range(attempts):
        try:
            os.replace(tmp, path)
            return
        except PermissionError:
            if i == attempts - 1:
                raise
            time.sleep(0.25)


def atomic_json(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")
    _replace(tmp, path)


def atomic_torch_save(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    _replace(tmp, path)


def atomic_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as fh:
        np.savez(fh, **arrays)
    _replace(tmp, path)


class EpisodeTracker:
    # Accumulates per-env episode return/length across vectorised steps.
    def __init__(self, n):
        self.returns = np.zeros(n, dtype=np.float64)
        self.lengths = np.zeros(n, dtype=np.int64)

    def step(self, rewards, dones):
        self.returns += rewards
        self.lengths += 1
        finished = []
        for i in np.flatnonzero(dones):
            finished.append((float(self.returns[i]), int(self.lengths[i])))
            self.returns[i] = 0.0
            self.lengths[i] = 0
        return finished


class RunMonitor:
    def __init__(self, run_tag, config, net, optimizer, resume_from=None):
        self.run_tag, self.config, self.net, self.optimizer = run_tag, config, net, optimizer
        self.run_dir = P2_ROOT / run_tag
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.frame = 0
        self.updates = 0
        self.samples_used = 0
        self.rewards, self.lengths, self.episode_frames, self.episode_wall = [], [], [], []
        self.best_mean = -math.inf
        self.best_saved_mean = -math.inf
        self.milestones = {}
        self.prior_elapsed = 0.0
        if resume_from is not None:
            saved = torch.load(resume_from, map_location=device, weights_only=False)
            if saved.get("algorithm") != config.algorithm:
                raise ValueError("Checkpoint algorithm does not match this run.")
            net.load_state_dict(saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            self.frame = int(saved["frame"])
            self.updates = int(saved["updates"])
            self.samples_used = int(saved["samples_used"])
            self.rewards = list(saved["rewards"])
            self.lengths = list(saved["lengths"])
            self.episode_frames = list(saved["episode_frames"])
            self.episode_wall = list(saved["episode_wall"])
            self.best_mean = float(saved["best_mean"])
            self.best_saved_mean = float(saved["best_saved_mean"])
            self.milestones = dict(saved.get("milestones", {}))
            self.prior_elapsed = float(saved["elapsed_seconds"])
            print(f"Resumed {run_tag} at frame {self.frame:,} ({len(self.rewards)} episodes).")
        self.writer = SummaryWriter(str(self.run_dir / "tensorboard"), purge_step=self.frame or None)
        self.started = time.perf_counter()
        self.session_start_frame = self.frame
        self.window_started, self.window_frame = self.started, self.frame
        self.last_print = -math.inf
        self.last_checkpoint_frame = self.frame
        self.last_diag_frame = -math.inf
        self.last_best_save_frame = -math.inf
        self.status = "running"
        self.n_params = count_params(net)

    def elapsed(self):
        return self.prior_elapsed + time.perf_counter() - self.started

    @property
    def solved(self):
        return self.status == "solved"

    def on_episode(self, episode_reward, episode_length):
        cfg, f, w = self.config, self.frame, self.writer
        wall_min = self.elapsed() / 60
        self.rewards.append(float(episode_reward))
        self.lengths.append(int(episode_length))
        self.episode_frames.append(int(f))
        self.episode_wall.append(float(wall_min))
        n_eps = len(self.rewards)
        mean_100 = float(np.mean(self.rewards[-100:]))
        w.add_scalar("episode/reward", episode_reward, f)
        w.add_scalar("episode/reward_100", mean_100, f)
        w.add_scalar("episode/length", episode_length, f)
        w.add_scalar("system/wall_minutes", wall_min, f)

        for name, hit in ((f"+{cfg.convergence_threshold:g}", mean_100 >= cfg.convergence_threshold),
                          ("solved", n_eps >= cfg.min_episodes_to_solve and mean_100 >= cfg.reward_bound)):
            if hit and name not in self.milestones:
                self.milestones[name] = {"frame": f, "episodes": n_eps, "wall_minutes": wall_min}
                print(f"*** {self.run_tag}: reached {name} at frame {f:,}, episode {n_eps}, {wall_min:.1f} min")

        if n_eps >= 10 and mean_100 > self.best_mean:
            self.best_mean = mean_100
        if (n_eps >= 10 and mean_100 > self.best_saved_mean
                and (f - self.last_best_save_frame >= cfg.best_save_interval or mean_100 >= cfg.reward_bound)):
            self.best_saved_mean = mean_100
            self.last_best_save_frame = f
            atomic_torch_save(self.checkpoint_payload(), self.run_dir / "best.pt")

        now = time.perf_counter()
        if now - self.last_print >= cfg.print_every_seconds:
            fps = (f - self.window_frame) / max(now - self.window_started, 1e-9)
            self.window_frame, self.window_started = f, now
            w.add_scalar("system/fps", fps, f)
            print(f"{self.run_tag} frame={f:>11,} episode={n_eps:>6} reward={episode_reward:6.1f} "
                  f"mean100={mean_100:7.3f} updates={self.updates:>8,} fps={fps:7.1f} "
                  f"wall={wall_min:6.1f}m")
            self.last_print = now

        if n_eps >= cfg.min_episodes_to_solve and mean_100 >= cfg.reward_bound:
            self.status = "solved"

    def on_update(self, samples, diagnostics):
        self.updates += 1
        self.samples_used += int(samples)
        if self.frame - self.last_diag_frame >= self.config.diagnostic_every:
            for key, value in diagnostics.items():
                self.writer.add_scalar(f"train/{key}", float(value), self.frame)
            self.last_diag_frame = self.frame
        if self.frame - self.last_checkpoint_frame >= self.config.checkpoint_every:
            self.save_latest()
            self.last_checkpoint_frame = self.frame

    def checkpoint_payload(self):
        return {
            "model": self.net.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "algorithm": self.config.algorithm,
            "frame": self.frame,
            "updates": self.updates,
            "samples_used": self.samples_used,
            "rewards": self.rewards,
            "lengths": self.lengths,
            "episode_frames": self.episode_frames,
            "episode_wall": self.episode_wall,
            "best_mean": self.best_mean,
            "best_saved_mean": self.best_saved_mean,
            "milestones": self.milestones,
            "elapsed_seconds": self.elapsed(),
            "config": asdict(self.config),
            "gpu": GPU_NAME,
        }

    def result_payload(self):
        elapsed = self.elapsed()
        return {
            "run_tag": self.run_tag,
            "algorithm": self.config.algorithm,
            "status": self.status,
            "frames": self.frame,
            "env_frames": 4 * self.frame,
            "episodes": len(self.rewards),
            "updates": self.updates,
            "samples_used": self.samples_used,
            "mean_reward_100": float(np.mean(self.rewards[-100:])) if self.rewards else None,
            "best_mean_reward_100": None if self.best_mean == -math.inf else self.best_mean,
            "milestones": self.milestones,
            "wall_clock_seconds": elapsed,
            "wall_clock_minutes": elapsed / 60,
            "avg_fps": self.frame / max(elapsed, 1e-9),
            "gpu": GPU_NAME,
            "cpu_count": CPU_COUNT,
            "num_params": self.n_params,
            "config": asdict(self.config),
        }

    def save_latest(self):
        atomic_torch_save(self.checkpoint_payload(), self.run_dir / "latest.pt")
        atomic_npz(self.run_dir / "episodes.npz",
                   frames=np.asarray(self.episode_frames, dtype=np.int64),
                   rewards=np.asarray(self.rewards, dtype=np.float32),
                   lengths=np.asarray(self.lengths, dtype=np.int64),
                   wall_minutes=np.asarray(self.episode_wall, dtype=np.float64))
        atomic_json(self.result_payload(), self.run_dir / "results.json")

    def finish(self, status):
        self.status = status
        self.save_latest()
        self.writer.flush()
        self.writer.close()
        print(json.dumps({k: v for k, v in self.result_payload().items() if k != "config"}, indent=2))

## 7. REINFORCE (Monte-Carlo policy gradient)

$$\nabla_\theta J(\theta)=\mathbb{E}_{\pi_\theta}\Big[\sum_t \nabla_\theta\log\pi_\theta(a_t\mid s_t)\,G_t\Big],\qquad G_t=\sum_{k\ge 0}\gamma^k r_{t+k}.$$

REINFORCE uses the actual sampled return $G_t$, with no critic and no bootstrapping. The gradient
is therefore **unbiased** but has **high variance**. The implementation uses the standard
variance-reduction steps for Pong:

1. **Per-point returns** (`point_reset=True`). Every Pong point ends with reward ±1, so the
   discounted sum restarts after each point. An action only gets credit for the rally it was part
   of. This is the approach from Karpathy's "Pong from Pixels". It does not change which policy is
   optimal, because points are independent.
2. **Return standardisation** within each batch. This works as a constant baseline and fixes the
   gradient scale.
3. **Entropy bonus** and **gradient-norm clipping**.

**Staying strictly on-policy.** Each update uses 16 envs × `rollout_steps` steps. Only steps
whose return is *complete* inside the rollout are used, meaning their rally ended before the
rollout ended. Each env's last, unfinished rally is dropped (about 10–15 % of steps at T = 512),
so no off-policy data is ever used. The batch can hold up to 8,192 states. The GPU processes it
in chunks of `update_chunk` states with gradient accumulation, so the result is exactly one
gradient step on the full batch.

In [ ]:
def discounted_returns_np(rewards, dones, gamma, point_reset):
    # rewards, dones: [T, N]. Returns G_t and a mask of steps whose return is complete in the rollout.
    T, N = rewards.shape
    returns = np.zeros((T, N), dtype=np.float32)
    mask = np.zeros((T, N), dtype=bool)
    running = np.zeros(N, dtype=np.float32)
    closed = np.zeros(N, dtype=bool)
    for t in range(T - 1, -1, -1):
        end = dones[t] | (rewards[t] != 0) if point_reset else dones[t].copy()
        running = rewards[t] + gamma * running * (~end)
        returns[t] = running
        closed |= end
        mask[t] = closed
    return returns, mask


def _pad_chunk(idx, start, chunk):
    # Fixed-shape GPU batches: with cudnn.benchmark every new shape triggers a slow re-tune.
    sel = idx[start:start + chunk]
    real = int(sel.numel())
    if real < chunk:
        sel = torch.cat([sel, sel[:1].expand(chunk - real)])
    return sel, real


def reinforce_update(net, optimizer, config, obs_buf, act_buf, rew_np, done_np, use_amp):
    returns_np, mask_np = discounted_returns_np(rew_np, done_np, config.gamma, config.point_reset)
    mask = torch.from_numpy(mask_np.reshape(-1)).to(device)
    idx = mask.nonzero(as_tuple=True)[0]
    n = int(idx.numel())
    if n < 2:
        return 0, None
    obs_flat = obs_buf.flatten(0, 1)
    act_flat = act_buf.flatten()
    G = torch.from_numpy(returns_np.reshape(-1)).to(device)[idx]
    raw_mean, raw_std = G.mean(), G.std(correction=0)
    chunk = min(config.update_chunk, obs_flat.shape[0])
    baseline = bool(config.value_baseline)

    if baseline:  # V(s) under the current parameters, before the update (a pure baseline: no bootstrap)
        values = torch.empty(n, device=device)
        with torch.no_grad():
            for start in range(0, n, chunk):
                sel, real = _pad_chunk(idx, start, chunk)
                _, v = forward_fp32(net, obs_flat[sel], use_amp)
                values[start:start + real] = v[:real]
        adv = G - values
    else:
        adv = G
    if config.normalize_returns:
        adv = (adv - adv.mean()) / (adv.std(correction=0) + 1e-8)

    optimizer.zero_grad(set_to_none=True)
    pg_total = torch.zeros((), device=device)
    ent_total = torch.zeros((), device=device)
    vl_total = torch.zeros((), device=device)
    for start in range(0, n, chunk):
        sel, real = _pad_chunk(idx, start, chunk)
        w = torch.zeros(chunk, device=device); w[:real] = 1.0
        a = torch.zeros(chunk, device=device); a[:real] = adv[start:start + real]
        logits, value = forward_fp32(net, obs_flat[sel], use_amp)
        logp_all = F.log_softmax(logits, dim=-1)
        logp = logp_all.gather(1, act_flat[sel].unsqueeze(1)).squeeze(1)
        entropy = -(logp_all.exp() * logp_all).sum(-1)
        pg_sum = -(logp * a * w).sum()
        ent_sum = (entropy * w).sum()
        loss = pg_sum - config.entropy_coef * ent_sum
        if baseline:
            g = torch.zeros(chunk, device=device); g[:real] = G[start:start + real]
            vl_sum = ((value - g).square() * w).sum()
            loss = loss + config.value_coef * vl_sum
            vl_total += vl_sum.detach()
        (loss / n).backward()
        pg_total += pg_sum.detach()
        ent_total += ent_sum.detach()
    grad_norm = nn.utils.clip_grad_norm_(net.parameters(), config.max_grad_norm)
    optimizer.step()
    diagnostics = {
        "policy_loss": pg_total / n,
        "entropy": ent_total / n,
        "grad_norm": grad_norm,
        "return_mean": raw_mean,
        "return_std": raw_std,
        "advantage_std": (G - values).std(correction=0) if baseline else raw_std,
        "batch_size": float(n),
        "used_fraction": n / mask_np.size,
    }
    if baseline:
        diagnostics.update({
            "value_loss": vl_total / n,
            "value_mean": values.mean(),
            "explained_variance": 1.0 - (G - values).var() / G.var().clamp_min(1e-8),
        })
    return n, diagnostics

## 8. A2C (synchronous advantage actor-critic)

The actor uses the advantage $A_t=\hat R_t - V(s_t)$ instead of the full Monte-Carlo return. The
critic $V_\phi$ is trained on the bootstrapped *n*-step target, or on GAE(λ) when `gae_lambda < 1`:

$$\hat R_t=\sum_{k=0}^{n-1}\gamma^k r_{t+k}+\gamma^n V_\phi(s_{t+n}),\qquad
\mathcal L=-\mathbb E[\log\pi(a_t|s_t)A_t]+c_v\,\mathbb E[(\hat R_t-V(s_t))^2]-c_e\,\mathcal H[\pi].$$

Bootstrapping lowers variance compared with REINFORCE, at the cost of some **bias** from the
critic's errors. That bias is measured in Section 14. All 16 environments step in lock-step, and
one update is made every `rollout_steps` steps. Episodes that end by time-limit truncation (rare
in Pong) get their final state's value added as a bootstrap, rather than being treated as
terminal.

In [ ]:
def compute_gae(rewards, dones, values, last_value, gamma, lam):
    # All tensors [T, N] (last_value [N]). dones[t]=1 means the episode ended at step t.
    T = rewards.shape[0]
    advantages = torch.zeros_like(rewards)
    gae = torch.zeros_like(last_value)
    for t in range(T - 1, -1, -1):
        next_value = last_value if t == T - 1 else values[t + 1]
        nonterminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae
    return advantages, advantages + values


def a2c_update(net, optimizer, config, obs_buf, act_buf, rew_np, done_np, val_buf, last_obs, use_amp):
    T, N = act_buf.shape
    rewards = torch.from_numpy(rew_np).to(device)
    dones = torch.from_numpy(done_np.astype(np.float32)).to(device)
    with torch.no_grad():
        _, last_value = forward_fp32(net, last_obs, use_amp)
        advantages, returns = compute_gae(rewards, dones, val_buf, last_value, config.gamma, config.gae_lambda)
    logits, values = forward_fp32(net, obs_buf.flatten(0, 1), use_amp)
    logp_all = F.log_softmax(logits, dim=-1)
    logp = logp_all.gather(1, act_buf.flatten().unsqueeze(1)).squeeze(1)
    entropy = -(logp_all.exp() * logp_all).sum(-1).mean()
    adv = advantages.flatten()
    ret = returns.flatten()
    policy_loss = -(logp * adv).mean()
    value_loss = F.mse_loss(values, ret)
    loss = policy_loss + config.value_coef * value_loss - config.entropy_coef * entropy

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(net.parameters(), config.max_grad_norm)
    optimizer.step()

    with torch.no_grad():
        explained_var = 1.0 - (ret - values).var() / ret.var().clamp_min(1e-8)
        diagnostics = {
            "loss": loss.detach(),
            "policy_loss": policy_loss.detach(),
            "value_loss": value_loss.detach(),
            "entropy": entropy.detach(),
            "grad_norm": grad_norm,
            "value_mean": values.mean(),        # critic estimate  (cf. DQN train/q_mean)
            "target_mean": ret.mean(),          # bootstrapped target (cf. DQN train/target_mean)
            "advantage_std": adv.std(),
            "explained_variance": explained_var,
        }
    return T * N, diagnostics

## 9. Shared vectorised trainer

The rollout and bookkeeping loop is the same for both algorithms. Only the update rule
(`reinforce_update` or `a2c_update`) and the rollout length differ.

In [ ]:
def train_pg(run_tag, config, resume_from=None):
    if config.algorithm not in {"reinforce", "a2c"}:
        raise ValueError(config.algorithm)
    set_seed(config.seed)
    envs = make_vec_env(config.num_envs, config.env_id)
    obs_shape = envs.single_observation_space.shape
    n_actions = int(envs.single_action_space.n)
    net = build_pg_net(config, obs_shape, n_actions)
    optimizer = build_optimizer(net, config)
    mon = RunMonitor(run_tag, config, net, optimizer, resume_from=resume_from)
    tracker = EpisodeTracker(config.num_envs)
    use_amp = bool(config.use_amp and AMP_DTYPE is not None)
    is_a2c = config.algorithm == "a2c"
    T, N = config.rollout_steps, config.num_envs

    # Rollout storage lives on the GPU (uint8 frames -> ~28 KB per state).
    obs_buf = torch.empty((T, N, *obs_shape), dtype=torch.uint8, device=device)
    act_buf = torch.empty((T, N), dtype=torch.long, device=device)
    val_buf = torch.empty((T, N), dtype=torch.float32, device=device) if is_a2c else None
    rew_np = np.zeros((T, N), dtype=np.float32)
    done_np = np.zeros((T, N), dtype=bool)

    obs, _ = envs.reset(seed=config.seed + mon.frame)
    obs_t = torch.from_numpy(obs).to(device)
    status = "error"
    try:
        while mon.frame < config.max_frames and not mon.solved:
            for t in range(T):
                obs_buf[t].copy_(obs_t)
                with torch.no_grad():
                    logits, value = forward_fp32(net, obs_t, use_amp)
                    actions = torch.distributions.Categorical(logits=logits).sample()
                act_buf[t] = actions
                if is_a2c:
                    val_buf[t] = value
                next_obs, rewards, terminated, truncated, infos = envs.step(actions.cpu().numpy())
                rewards = np.asarray(rewards, dtype=np.float32)
                dones = np.logical_or(terminated, truncated)
                stored = rewards
                if is_a2c:
                    trunc_only = np.logical_and(truncated, np.logical_not(terminated))
                    if trunc_only.any():  # bootstrap time-limit truncations from V(final_obs)
                        ids = np.flatnonzero(trunc_only)
                        final = np.zeros((N, *obs_shape), dtype=np.uint8)  # fixed shape for cuDNN
                        for i in ids:
                            final[i] = infos["final_obs"][i]
                        with torch.no_grad():
                            _, v_final = forward_fp32(net, torch.from_numpy(final).to(device), use_amp)
                        stored = rewards.copy()
                        stored[ids] += config.gamma * v_final.cpu().numpy()[ids]
                rew_np[t] = stored
                done_np[t] = dones
                mon.frame += N
                for ep_reward, ep_len in tracker.step(rewards, dones):
                    mon.on_episode(ep_reward, ep_len)
                obs_t = torch.from_numpy(next_obs).to(device, non_blocking=True)
                if mon.solved:
                    break
            if mon.solved:
                break
            if config.lr_schedule == "linear":
                lr_now = config.learning_rate * max(1.0 - mon.frame / config.max_frames, 0.0)
                for group in optimizer.param_groups:
                    group["lr"] = lr_now
            if is_a2c:
                samples, diag = a2c_update(net, optimizer, config, obs_buf, act_buf,
                                           rew_np, done_np, val_buf, obs_t, use_amp)
            else:
                samples, diag = reinforce_update(net, optimizer, config, obs_buf, act_buf,
                                                 rew_np, done_np, use_amp)
            if diag is not None:
                diag["learning_rate"] = optimizer.param_groups[0]["lr"]
                for v in diag.values():
                    if torch.is_tensor(v) and not torch.isfinite(v).all():
                        raise RuntimeError(f"Non-finite diagnostic at frame {mon.frame:,}")
                mon.on_update(samples, diag)
        status = "solved" if mon.solved else "max_frames"
    except KeyboardInterrupt:
        status = "interrupted"
        print("Interrupted; checkpoint saved. Re-run the cell to resume.")
    finally:
        mon.finish(status)
        envs.close()
    return mon.result_payload()


FORCE_RESTART = False  # True deletes the run folder and trains from scratch


def run_or_resume(run_tag, config):
    run_dir = P2_ROOT / run_tag
    if FORCE_RESTART and run_dir.exists():
        print(f"FORCE_RESTART: deleting {run_dir}")
        shutil.rmtree(run_dir)
    results_path, latest = run_dir / "results.json", run_dir / "latest.pt"
    if results_path.exists():
        prev = json.loads(results_path.read_text(encoding="utf-8"))
        if prev.get("status") == "solved" or (
            prev.get("status") == "max_frames" and prev.get("frames", 0) >= config.max_frames
        ):
            print(f"Using completed run: {run_tag} ({prev['status']}, {prev['frames']:,} frames)")
            return prev
    return train_pg(run_tag, config, resume_from=latest if latest.exists() else None)

## 10. Correctness checks (run these before any long training)

* The return and GAE calculations are checked against hand-computed values.
* A gradient-direction check confirms that the policy-gradient loss raises the log-probability of
  positive-advantage actions and lowers it for negative-advantage actions.
* A critic check confirms that the value loss can fit fixed targets.
* Two smoke runs of about 30k frames exercise the full vectorised loop, checkpointing and logging.
* The separate-critic check (experiment A8) confirms that the value loss does not reach the policy's
  torso, and that the default shared network still loads the earlier A2C checkpoint.

In [ ]:
def correctness_checks():
    g = 0.99
    # --- REINFORCE per-point returns ---
    r = np.array([[0], [0], [1], [0], [-1], [0]], dtype=np.float32)
    d = np.zeros_like(r, dtype=bool)
    G, m = discounted_returns_np(r, d, g, point_reset=True)
    assert np.allclose(G[:5, 0], [g * g, g, 1.0, -g, -1.0]), G[:, 0]
    assert m[:, 0].tolist() == [True] * 5 + [False]
    _, m2 = discounted_returns_np(r, d, g, point_reset=False)
    assert not m2.any(), "without a terminal no full-episode return is complete"
    d2 = d.copy(); d2[5] = True
    G3, m3 = discounted_returns_np(r, d2, g, point_reset=False)
    assert m3.all() and np.isclose(G3[0, 0], g ** 2 - g ** 4)

    # --- GAE(lambda=1) equals the n-step bootstrapped return ---
    torch.manual_seed(0)
    T, N = 5, 3
    rew, val, last = torch.randn(T, N), torch.randn(T, N), torch.randn(N)
    dn = torch.zeros(T, N); dn[2, 1] = 1.0
    _, ret = compute_gae(rew, dn, val, last, g, 1.0)
    for n_ in range(N):
        for t in range(T):
            acc, disc, ended = 0.0, 1.0, False
            for k in range(t, T):
                acc += disc * rew[k, n_].item()
                if dn[k, n_] > 0:
                    ended = True
                    break
                disc *= g
            if not ended:
                acc += disc * last[n_].item()
            assert abs(acc - ret[t, n_].item()) < 1e-4

    # --- policy-gradient direction ---
    set_seed(SEED)
    net = PGNet((4, 84, 84), 6, with_value=True).to(device)
    opt = optim.Adam(net.parameters(), lr=1e-4)
    obs = torch.randint(0, 256, (64, 4, 84, 84), dtype=torch.uint8, device=device)
    acts = torch.randint(0, 6, (64,), device=device)
    adv = torch.where(acts < 3, 1.0, -1.0)  # actions 0-2 good, 3-5 bad

    def logp():
        logits, _ = forward_fp32(net, obs, False)
        return F.log_softmax(logits, -1).gather(1, acts[:, None]).squeeze(1)

    before = logp().detach()
    for _ in range(5):  # a few small steps: the PG objective is unbounded, so don't over-train
        loss = -(logp() * adv).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    delta = logp().detach() - before
    assert delta[adv > 0].mean() > 0 > delta[adv < 0].mean(), "PG loss moves log-probs the wrong way"

    # --- critic can fit fixed targets ---
    targets = torch.randn(64, device=device)
    opt = optim.Adam(net.parameters(), lr=1e-3)
    losses = []
    for _ in range(60):
        _, v = forward_fp32(net, obs, False)
        vl = F.mse_loss(v, targets)
        opt.zero_grad(); vl.backward(); opt.step()
        losses.append(vl.item())
    assert losses[-1] < 0.5 * losses[0], losses
    print("Returns, GAE, policy-gradient direction and critic-fit checks passed.")


correctness_checks()

In [ ]:
@contextlib.contextmanager
def exact_fp32():
    # TF32 (enabled above for speed on RTX 30xx/40xx/50xx) keeps ~10 mantissa bits, so two batch shapes
    # can differ by ~1e-3 relative. Equivalence checks must compare exact float32 arithmetic instead.
    saved = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32,
             torch.backends.cudnn.benchmark, torch.get_float32_matmul_precision())
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.set_float32_matmul_precision("highest")
    try:
        yield
    finally:
        (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32,
         torch.backends.cudnn.benchmark) = saved[:3]
        torch.set_float32_matmul_precision(saved[3])


def baseline_variant_check():
    # The value-baseline REINFORCE update must give the same gradient with or without chunk padding.
    T, N = 64, 4
    g = torch.Generator().manual_seed(0)
    obs = torch.randint(0, 256, (T, N, 4, 84, 84), dtype=torch.uint8, generator=g).to(device)
    act = torch.randint(0, 6, (T, N), generator=g).to(device)
    rng = np.random.default_rng(0)
    rew = np.zeros((T, N), np.float32)
    rew[rng.random((T, N)) < 0.05] = 1.0
    rew[rng.random((T, N)) < 0.05] = -1.0
    done = np.zeros((T, N), bool)
    grads = []
    with exact_fp32():
        for chunk in (T * N, 37):
            torch.manual_seed(1)
            cfg = replace(REINFORCE_CONFIG, value_baseline=True, value_coef=0.5, update_chunk=chunk, max_grad_norm=1e9)
            net = build_pg_net(cfg, (4, 84, 84), 6)
            opt = optim.SGD(net.parameters(), lr=0.0)
            n, diag = reinforce_update(net, opt, cfg, obs, act, rew, done, False)
            assert n > 0 and all(torch.isfinite(torch.as_tensor(v)).all() for v in diag.values())
            grads.append(torch.cat([p.grad.flatten() for p in net.parameters()]))
    max_diff = float((grads[0] - grads[1]).abs().max())
    scale = float(grads[0].abs().max())
    # Only summation order differs between the two batchings, so fp32 agreement should be ~1e-6 relative.
    assert max_diff <= 1e-4 * scale + 1e-8, f"max |diff| {max_diff:.3e} vs gradient scale {scale:.3e}"
    print(f"Value-baseline REINFORCE: padded and unpadded gradients match "
          f"(max |diff| {max_diff:.2e}, {max_diff / max(scale, 1e-12):.1e} relative; TF32 restored for training).")


baseline_variant_check()

In [ ]:
def separate_critic_check():
    shared = build_pg_net(A2C_CONFIG, (4, 84, 84), 6)
    separate = build_pg_net(replace(A2C_CONFIG, shared_torso=False), (4, 84, 84), 6)
    assert not any(k.startswith("value_") and k != "value_head.weight" and k != "value_head.bias"
                   for k in shared.state_dict()), "shared network must keep the earlier layout"
    assert count_params(separate) > count_params(shared)
    obs = torch.randint(0, 256, (8, 4, 84, 84), dtype=torch.uint8, device=device)
    logits, value = forward_fp32(separate, obs, False)
    assert logits.shape == (8, 6) and value.shape == (8,)
    separate.zero_grad(set_to_none=True)
    value.sum().backward()
    leaked = [n for n, p in separate.named_parameters()
              if (n.startswith("features") or n.startswith("fc.") or n.startswith("policy_head"))
              and p.grad is not None and float(p.grad.abs().sum()) > 0]
    assert not leaked, f"value gradient reached the policy network: {leaked}"
    ckpt = P2_ROOT / "a2c_seed42" / "best.pt"
    if ckpt.exists():
        shared.load_state_dict(torch.load(ckpt, map_location=device, weights_only=False)["model"])
        print("Earlier A2C checkpoint loads into the shared network.")
    print(f"Separate-critic check passed ({count_params(shared):,} vs {count_params(separate):,} parameters).")


separate_critic_check()

In [ ]:
RUN_SMOKE_TESTS = TRAINING_AVAILABLE  # ~1-2 min: runs the new options end to end before the long queue

if RUN_SMOKE_TESTS:
    for tag, cfg in [
        ("smoke_a2c_separate_critic", replace(A2C_CONFIG, rollout_steps=20, shared_torso=False, max_frames=20_000,
                                              diagnostic_every=1, print_every_seconds=10)),
        ("smoke_reinforce_T128_nonorm", replace(REINFORCE_CONFIG, rollout_steps=128, normalize_returns=False,
                                                max_frames=16 * 128 * 8, diagnostic_every=1, print_every_seconds=10)),
    ]:
        shutil.rmtree(P2_ROOT / tag, ignore_errors=True)
        res = train_pg(tag, cfg)
        assert res["frames"] >= cfg.max_frames and res["updates"] > 0, res
        print(f"{tag}: OK  ({res['avg_fps']:.0f} agent steps/s, {res['updates']} updates)\n")
else:
    print("Smoke tests skipped (no GPU).")

## 11. TensorBoard

Project 2 runs (`reinforce_seed42`, `a2c_seed42`, sweep trials) appear next to the Project 1 DQN
runs. They share the same `episode/reward_100` tag, so the curves overlay directly. Useful A2C tags
are `train/value_mean` against `train/target_mean`, which mirror DQN's `train/q_mean` against
`train/target_mean`, and `train/explained_variance`. For REINFORCE, look at `train/return_std`,
`train/grad_norm` and `train/entropy`.

In [ ]:
SHOW_TENSORBOARD = TRAINING_AVAILABLE  # set True to open the dashboard in analysis-only mode too

_specs = [f"p2:{P2_ROOT.as_posix()}"]
for name, tag in [("dqn_baseline", "baseline_epsilon_seed42"), ("dqn_noisynet", "noisynet_seed42"),
                  ("dqn_noisynet_per", "per_noisynet_seed42"), ("dqn_boltzmann_tuned", "boltzmann_softmax_decay100k_seed42")]:
    if (P1_ROOT / tag / "tensorboard").exists():
        _specs.append(f"{name}:{(P1_ROOT / tag / 'tensorboard').as_posix()}")
TB_SPEC = ",".join(_specs)
print(TB_SPEC)
if SHOW_TENSORBOARD:
    %load_ext tensorboard
    %tensorboard --logdir_spec "$TB_SPEC" --port 6006
    # The dashboard also opens in a browser at http://localhost:6006

## Helper functions (curves, AUC, evaluation)

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


def tensorboard_curve(event_dir, scalar="episode/reward_100"):
    event_dir = Path(event_dir)
    if not event_dir.exists():
        return np.array([]), np.array([])
    acc = EventAccumulator(str(event_dir), size_guidance={"scalars": 0})
    acc.Reload()
    if scalar not in acc.Tags().get("scalars", []):
        return np.array([]), np.array([])
    by_step = {e.step: e.value for e in acc.Scalars(scalar)}
    steps = np.asarray(sorted(by_step), dtype=np.int64)
    return steps, np.asarray([by_step[s] for s in steps], dtype=np.float64)


def p2_episode_log(run_tag):
    path = P2_ROOT / run_tag / "episodes.npz"
    if not path.exists():
        return None
    d = np.load(path)
    df = pd.DataFrame({"frame": d["frames"], "reward": d["rewards"], "wall_minutes": d["wall_minutes"]})
    df["reward_100"] = df.reward.rolling(100, min_periods=1).mean()
    return df


def normalized_auc(frames, reward_100, horizon):
    # Map Pong reward [-21, 21] -> [0, 1] and average over [0, horizon] (Project 1 definition).
    frames, reward_100 = np.asarray(frames), np.asarray(reward_100)
    if len(frames) == 0:
        return float("nan")
    grid = np.linspace(0, horizon, 2001)
    y = np.interp(grid, frames, reward_100, left=reward_100[0], right=reward_100[-1])
    y = np.clip((y + 21.0) / 42.0, 0.0, 1.0)
    return float(np.trapezoid(y, grid) / max(horizon, 1))


def run_sweep(prefix, base_config, grid, trial_frames):
    rows = []
    for name, overrides in grid.items():
        cfg = replace(base_config, **overrides, max_frames=trial_frames)
        tag = f"sweep_{prefix}_{name}"
        print(f"\n===== {tag}: {overrides} =====")
        res = run_or_resume(tag, cfg)
        log = p2_episode_log(tag)
        auc = normalized_auc(log.frame, log.reward_100, trial_frames) if log is not None else float("nan")
        rows.append({"trial": name, "run_tag": tag, **overrides,
                     "status": res["status"], "frames": res["frames"],
                     "final_reward_100": res["mean_reward_100"],
                     "best_reward_100": res["best_mean_reward_100"],
                     "normalized_auc": auc,
                     "minutes": round(res["wall_clock_minutes"], 1),
                     "fps": round(res["avg_fps"], 1)})
    df = pd.DataFrame(rows).sort_values(["final_reward_100", "normalized_auc"], ascending=False)
    df = df.reset_index(drop=True)
    df.to_csv(P2_ROOT / f"sweep_{prefix}.csv", index=False)
    display(df)
    return df


def choose_config(prefix, base_config, grid, sweep_df, full_frames):
    choice_path = P2_ROOT / f"chosen_{prefix}.json"
    if sweep_df is not None and len(sweep_df):
        winner = sweep_df.iloc[0]["trial"]
        atomic_json({"trial": winner, "overrides": grid[winner]}, choice_path)
    elif choice_path.exists():
        winner = json.loads(choice_path.read_text(encoding="utf-8"))["trial"]
    else:
        winner = None
    overrides = grid.get(winner, {}) if winner else {}
    cfg = replace(base_config, **overrides, max_frames=full_frames)
    print(f"{prefix}: chosen trial = {winner or 'defaults'} -> {overrides}")
    return cfg, winner


def promote_trial(prefix, winner, full_tag):
    # Continue the winning trial as the full run (copies checkpoints + TensorBoard + episode log).
    if winner is None:
        return
    src, dst = P2_ROOT / f"sweep_{prefix}_{winner}", P2_ROOT / full_tag
    if dst.exists() or not (src / "latest.pt").exists():
        return
    shutil.copytree(src, dst)
    print(f"Promoted {src.name} -> {full_tag}; training continues from frame "
          f"{json.loads((dst / 'results.json').read_text(encoding='utf-8'))['frames']:,}")

In [ ]:
EVAL_EPISODES = 30
RERUN_EVAL = False
EVAL_GREEDY = False  # greedy PG evaluation can play very long games; stochastic is the trained policy


def discounted_episode_returns(rewards, gamma=GAMMA):
    out, running = np.zeros(len(rewards), dtype=np.float64), 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = rewards[t] + gamma * running
        out[t] = running
    return out


@torch.no_grad()
def play_and_measure(policy_fn, episodes=EVAL_EPISODES, seed_offset=20_000):
    env = make_env()
    scores, lengths, estimates, mc_returns = [], [], [], []
    for ep in range(episodes):
        state, _ = env.reset(seed=SEED + seed_offset + ep)
        ep_rewards, ep_est, done = [], [], False
        while not done:
            action, estimate = policy_fn(state, env)
            state, r, terminated, truncated, _ = env.step(action)
            ep_rewards.append(float(r))
            ep_est.append(estimate)
            done = bool(terminated or truncated)
        scores.append(sum(ep_rewards))
        lengths.append(len(ep_rewards))
        estimates.extend(ep_est)
        mc_returns.extend(discounted_episode_returns(ep_rewards))
    env.close()
    est, mc = np.asarray(estimates, dtype=np.float64), np.asarray(mc_returns, dtype=np.float64)
    result = {"episodes": episodes, "mean": float(np.mean(scores)), "std": float(np.std(scores)),
              "min": float(np.min(scores)), "max": float(np.max(scores)), "scores": scores,
              "mean_length": float(np.mean(lengths))}
    finite = np.isfinite(est)
    if finite.any():
        err = est[finite] - mc[finite]
        rng = np.random.default_rng(0)
        keep = rng.choice(finite.sum(), size=min(4000, finite.sum()), replace=False)
        result.update({
            "estimate_mean": float(est[finite].mean()),
            "mc_return_mean": float(mc[finite].mean()),
            "bias_mean": float(err.mean()),
            "abs_error_mean": float(np.abs(err).mean()),
            "corr": float(np.corrcoef(est[finite], mc[finite])[0, 1]) if finite.sum() > 2 else float("nan"),
            "overestimated_fraction": float((err > 0).mean()),
            "sample_estimates": est[finite][keep].tolist(),
            "sample_returns": mc[finite][keep].tolist(),
        })
    return result


def load_dqn(run_tag):
    path = P1_ROOT / run_tag / "best.pt"
    if not path.exists():
        return None
    saved = torch.load(path, map_location=device, weights_only=False)
    algo = saved.get("algorithm", "epsilon")
    net = NoisyDQN((4, 84, 84), 6) if algo == "noisynet" else DQN((4, 84, 84), 6)
    net.load_state_dict(saved["online"])
    return net.to(device).eval()


def load_pg(run_tag):
    path = P2_ROOT / run_tag / "best.pt"
    if not path.exists():
        return None
    saved = torch.load(path, map_location=device, weights_only=False)
    with_value = saved["algorithm"] == "a2c" or bool(saved.get("config", {}).get("value_baseline", False))
    net = PGNet((4, 84, 84), 6, with_value=with_value,
                shared_torso=bool(saved.get("config", {}).get("shared_torso", True)))
    net.load_state_dict(saved["model"])
    return net.to(device).eval()


def dqn_policy(net, epsilon, seed):
    rng = np.random.default_rng(seed)
    def fn(state, env):
        q = net(torch.from_numpy(state).unsqueeze(0).to(device)).squeeze(0)
        a = int(q.argmax().item()) if rng.random() >= epsilon else int(rng.integers(env.action_space.n))
        return a, float(q[a].item())
    return fn


def pg_policy(net, stochastic, seed):
    gen = torch.Generator(device=device).manual_seed(seed)
    def fn(state, env):
        logits, value = forward_fp32(net, torch.from_numpy(state).unsqueeze(0).to(device), False)
        if stochastic:
            a = int(torch.multinomial(torch.softmax(logits, -1), 1, generator=gen).item())
        else:
            a = int(logits.argmax(-1).item())
        return a, (float(value.item()) if value is not None else float("nan"))
    return fn

## 12. Experiment plan

**Reference configurations.** Each new experiment changes **one setting** of the best configuration found
so far, and keeps everything else (seed 42, environment, network, frame cap) fixed:

* **REINFORCE reference** = `reinforce_T256_seed42`: Adam, lr 5e-4, rollout 256 steps × 16 envs, entropy 0.01,
  per-point returns, standardised returns. It solved Pong at 14.83M frames (the first tuned config,
  rollout 512, needed 17.83M).
* **A2C reference** = `a2c_n20_seed42`: RMSprop, lr 3.5e-4, 20-step returns, 16 envs, entropy 0.01,
  value coefficient 0.5, shared torso. It solved at 3.40M frames in 19.4 min (the first tuned config,
  5-step returns, needed 3.89M frames and 33 min).

Frame caps: **20M frames per REINFORCE run** and **10M per A2C run**; every run stops as soon as it solves.

### Earlier experiments (done, never retrained)

| ID | Experiment | Runs |
|---|---|---|
| E1 | REINFORCE: first tuned config and three variants | T=512 (tuned), T=256, lr 1e-3, learned V(s) baseline |
| E2 | A2C: first tuned config (n=5) over three seeds | seeds 42, 43, 44 |
| E3 | A2C: variants of the first tuned config | lr 7e-4 linear decay, n=20, 32 envs, no entropy bonus |

### New REINFORCE experiments

| ID | Experiment | New runs | Question |
|---|---|---|---|
| **R1** | Seeds of the reference (T=256) | seeds 43, 44 | Is the best REINFORCE result robust, or a lucky seed? |
| **R2** | Rollout length | T=128 | 512 → 256 helped. Do even more frequent, smaller updates help further, or does the gradient become too noisy? |
| **R3** | Learning rate at T=256 | lr 2.5e-4, 7e-4 | Where is the best learning rate between "works" (5e-4) and "fails" (1e-3)? |
| **R4** | Ablation: return standardisation off | `normalize_returns=False` | How much does the batch baseline (standardising returns) matter for REINFORCE's variance? |

These four were chosen because R1 makes the main REINFORCE result statistically defensible, R2 and R3 are
the two settings REINFORCE is most sensitive to, and R4 directly supports the root-cause discussion
(variance). The other candidates were left out: the entropy bonus never limited REINFORCE (entropy stays
high for most of training); full-episode returns need rollouts longer than a Pong game (> 2,000 steps),
so very few returns would be complete; and the learned baseline was already tested in E1.

### New A2C experiments

| ID | Experiment | New runs | Question |
|---|---|---|---|
| **A1** | Seeds of the reference (n=20) | seeds 43, 44 | Is the best A2C result robust? |
| **A2** | n-step length | n = 10, 40, 80 | Bias-variance trade-off of the bootstrapped target (n=5 and n=20 already done) |
| **A3** | GAE λ at n=20 | λ = 0.9, 0.95, 0.98 | Does mixing n-step returns (GAE) beat plain 20-step returns (λ=1)? |
| **A4** | Entropy coefficient at n=20 | 0.005, 0.02, 0.05 | How much exploration bonus is best? (0 collapsed in E3) |
| **A5** | Learning rate at n=20 | 2.5e-4, 5e-4, 1e-3 | Does the longer rollout change the best learning rate? |
| **A6** | Optimiser | Adam (same lr) | RMSprop vs Adam in isolation (the first sweep changed Adam together with n and λ) |
| **A7** | Value-loss coefficient | 0.25, 1.0 | Balance between training the critic and the actor |
| **A8** | Separate actor and critic networks | `shared_torso=False` | Does sharing the CNN between policy and value help or hurt? |

**Estimated compute** on an RTX laptop with 16+ CPU threads: about 1.2–1.8 h per REINFORCE run (6 runs) and
about 20–60 min per A2C run (18 runs), roughly **15–25 hours** in total. The queue runs the experiments in
priority order (`RUN_ORDER`): the seed experiments first, then the most informative sweeps.

In [ ]:
def chosen_overrides(prefix, default):
    path = P2_ROOT / f"chosen_{prefix}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))["overrides"]
    return default


def load_results(tag):
    path = P2_ROOT / tag / "results.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


# Configurations picked by the first-run sweeps (exactly as in the earlier notebooks).
REINFORCE_BEST = replace(REINFORCE_CONFIG, **chosen_overrides("reinforce", {"learning_rate": 5e-4, "rollout_steps": 512}))
A2C_BEST = replace(A2C_CONFIG, **chosen_overrides("a2c", {"learning_rate": 3.5e-4}))

# Reference configurations of the new experiments: the best run of each algorithm so far.
REINFORCE_REF = replace(REINFORCE_BEST, rollout_steps=256)   # = reinforce_T256_seed42
A2C_REF = replace(A2C_BEST, rollout_steps=20)                # = a2c_n20_seed42
REINFORCE_FRAMES = 20_000_000   # cap per REINFORCE run (stops when solved)
A2C_FRAMES = 10_000_000         # cap per A2C run (stops when solved)


def rf(**overrides):
    return replace(REINFORCE_REF, max_frames=REINFORCE_FRAMES, **overrides)


def ac(**overrides):
    return replace(A2C_REF, max_frames=A2C_FRAMES, **overrides)


# Runs finished in the earlier notebooks. They are read from CMPE260_Project2/ and never retrained.
EARLIER_RUNS = {
    "reinforce_seed42":           "REINFORCE T=512, lr 5e-4 (first tuned config)",
    "reinforce_T256_seed42":      "REINFORCE T=256 (reference of R1-R4)",
    "reinforce_lr1e-3_seed42":    "REINFORCE lr 1e-3 (T=512)",
    "reinforce_vbaseline_seed42": "REINFORCE + learned V(s) baseline (T=512)",
    "a2c_seed42":                 "A2C n=5 (first tuned config), seed 42",
    "a2c_seed43":                 "A2C n=5, seed 43",
    "a2c_seed44":                 "A2C n=5, seed 44",
    "a2c_lin7e-4_seed42":         "A2C n=5, lr 7e-4 with linear decay",
    "a2c_n20_seed42":             "A2C n=20 (reference of A1-A8)",
    "a2c_32envs_seed42":          "A2C n=5, 32 envs",
    "a2c_noent_seed42":           "A2C n=5, no entropy bonus",
}

# New runs trained by this notebook: tag -> (config, change from the reference).
NEW_RUNS = {
    # REINFORCE (reference: reinforce_T256_seed42)
    "reinforce_T256_seed43":          (rf(seed=43), "seed 43"),
    "reinforce_T256_seed44":          (rf(seed=44), "seed 44"),
    "reinforce_T128_seed42":          (rf(rollout_steps=128), "rollout 128"),
    "reinforce_T256_lr2.5e-4_seed42": (rf(learning_rate=2.5e-4), "lr 2.5e-4"),
    "reinforce_T256_lr7e-4_seed42":   (rf(learning_rate=7e-4), "lr 7e-4"),
    "reinforce_T256_nonorm_seed42":   (rf(normalize_returns=False), "no return standardisation"),
    # A2C (reference: a2c_n20_seed42)
    "a2c_n20_seed43":                 (ac(seed=43), "seed 43"),
    "a2c_n20_seed44":                 (ac(seed=44), "seed 44"),
    "a2c_n10_seed42":                 (ac(rollout_steps=10), "n=10"),
    "a2c_n40_seed42":                 (ac(rollout_steps=40), "n=40"),
    "a2c_n80_seed42":                 (ac(rollout_steps=80), "n=80"),
    "a2c_n20_gae0.9_seed42":          (ac(gae_lambda=0.9), "GAE λ=0.9"),
    "a2c_n20_gae0.95_seed42":         (ac(gae_lambda=0.95), "GAE λ=0.95"),
    "a2c_n20_gae0.98_seed42":         (ac(gae_lambda=0.98), "GAE λ=0.98"),
    "a2c_n20_ent0.005_seed42":        (ac(entropy_coef=0.005), "entropy 0.005"),
    "a2c_n20_ent0.02_seed42":         (ac(entropy_coef=0.02), "entropy 0.02"),
    "a2c_n20_ent0.05_seed42":         (ac(entropy_coef=0.05), "entropy 0.05"),
    "a2c_n20_lr2.5e-4_seed42":        (ac(learning_rate=2.5e-4), "lr 2.5e-4"),
    "a2c_n20_lr5e-4_seed42":          (ac(learning_rate=5e-4), "lr 5e-4"),
    "a2c_n20_lr1e-3_seed42":          (ac(learning_rate=1e-3), "lr 1e-3"),
    "a2c_n20_adam_seed42":            (ac(optimizer="adam"), "Adam instead of RMSprop"),
    "a2c_n20_vcoef0.25_seed42":       (ac(value_coef=0.25), "value coef 0.25"),
    "a2c_n20_vcoef1_seed42":          (ac(value_coef=1.0), "value coef 1.0"),
    "a2c_n20_separate_seed42":        (ac(shared_torso=False), "separate critic network"),
}

# Each experiment lists its runs, reference first. Runs can appear in several experiments.
EXPERIMENTS = {
    "E1": dict(title="REINFORCE: first tuned config and variants (earlier)",
               runs=["reinforce_seed42", "reinforce_T256_seed42", "reinforce_lr1e-3_seed42", "reinforce_vbaseline_seed42"]),
    "E2": dict(title="A2C n=5 over three seeds (earlier)", runs=["a2c_seed42", "a2c_seed43", "a2c_seed44"]),
    "E3": dict(title="A2C variants of the n=5 config (earlier)",
               runs=["a2c_seed42", "a2c_lin7e-4_seed42", "a2c_n20_seed42", "a2c_32envs_seed42", "a2c_noent_seed42"]),
    "R1": dict(title="REINFORCE: seeds of the reference (T=256)",
               runs=["reinforce_T256_seed42", "reinforce_T256_seed43", "reinforce_T256_seed44"]),
    "R2": dict(title="REINFORCE: rollout length (T)",
               runs=["reinforce_T256_seed42", "reinforce_T128_seed42", "reinforce_seed42"]),
    "R3": dict(title="REINFORCE: learning rate (T=256; lr 1e-3 was run at T=512)",
               runs=["reinforce_T256_seed42", "reinforce_T256_lr2.5e-4_seed42", "reinforce_T256_lr7e-4_seed42",
                     "reinforce_lr1e-3_seed42"]),
    "R4": dict(title="REINFORCE ablation: return standardisation",
               runs=["reinforce_T256_seed42", "reinforce_T256_nonorm_seed42"]),
    "A1": dict(title="A2C: seeds of the reference (n=20)", runs=["a2c_n20_seed42", "a2c_n20_seed43", "a2c_n20_seed44"]),
    "A2": dict(title="A2C: n-step length", runs=["a2c_n20_seed42", "a2c_seed42", "a2c_n10_seed42", "a2c_n40_seed42", "a2c_n80_seed42"]),
    "A3": dict(title="A2C: GAE λ (n=20)", runs=["a2c_n20_seed42", "a2c_n20_gae0.9_seed42", "a2c_n20_gae0.95_seed42", "a2c_n20_gae0.98_seed42"]),
    "A4": dict(title="A2C: entropy coefficient (n=20; 'no entropy' was run at n=5)",
               runs=["a2c_n20_seed42", "a2c_n20_ent0.005_seed42", "a2c_n20_ent0.02_seed42", "a2c_n20_ent0.05_seed42", "a2c_noent_seed42"]),
    "A5": dict(title="A2C: learning rate (n=20)", runs=["a2c_n20_seed42", "a2c_n20_lr2.5e-4_seed42", "a2c_n20_lr5e-4_seed42", "a2c_n20_lr1e-3_seed42"]),
    "A6": dict(title="A2C: optimiser (n=20)", runs=["a2c_n20_seed42", "a2c_n20_adam_seed42"]),
    "A7": dict(title="A2C: value-loss coefficient (n=20)", runs=["a2c_n20_seed42", "a2c_n20_vcoef0.25_seed42", "a2c_n20_vcoef1_seed42"]),
    "A8": dict(title="A2C: shared vs separate actor-critic network (n=20)", runs=["a2c_n20_seed42", "a2c_n20_separate_seed42"]),
}

# Training order of the new experiments (priority first) and on/off switches.
RUN_ORDER = ["A1", "R1", "A2", "R2", "A3", "A4", "R3", "A5", "A6", "A7", "A8", "R4"]
RUN_EXPERIMENTS = {exp_id: True for exp_id in RUN_ORDER}   # e.g. set the R* to False on an A2C-only machine

# ---- consistency checks: no tag is both earlier and new; every listed run is defined ----
assert not set(EARLIER_RUNS) & set(NEW_RUNS)
for exp_id, exp in EXPERIMENTS.items():
    for tag in exp["runs"]:
        assert tag in EARLIER_RUNS or tag in NEW_RUNS, f"{exp_id}: unknown run {tag}"
IGNORED_FIELDS = {"max_frames", "checkpoint_every", "diagnostic_every", "best_save_interval",
                  "print_every_seconds", "update_chunk", "use_amp"}


def config_matches(saved, cfg):
    # True when a saved results.json config describes the same experiment as cfg.
    default = asdict(cfg)
    return all(saved.get(k, asdict(PGConfig(algorithm=cfg.algorithm))[k]) == v
               for k, v in default.items() if k not in IGNORED_FIELDS)


for ref_tag, ref_cfg in (("reinforce_T256_seed42", rf()), ("a2c_n20_seed42", ac())):
    saved = load_results(ref_tag)
    if saved is not None:
        assert config_matches(saved["config"], ref_cfg), f"reference config differs from the saved {ref_tag}"
print(f"{len(EARLIER_RUNS)} earlier runs, {len(NEW_RUNS)} new runs, {len(EXPERIMENTS)} experiments. "
      "Reference configs match the saved reference runs.")

In [ ]:
def finished(results, cfg):
    return results["status"] == "solved" or (results["status"] == "max_frames" and results["frames"] >= cfg.max_frames)


def run_status(tag):
    r = load_results(tag)
    if tag in EARLIER_RUNS:
        return "done (earlier)" if r else "MISSING (copy its folder into CMPE260_Project2/)"
    cfg = NEW_RUNS[tag][0]
    if r is None:
        return "to run"
    if not config_matches(r["config"], cfg):
        return "CONFLICT (folder holds a different config)"
    if finished(r, cfg):
        return "done"
    return "partial (resumes)" if (P2_ROOT / tag / "latest.pt").exists() else "partial (no latest.pt: restarts)"


def estimated_fps(cfg):
    # Measured on the RTX 5080 laptop: REINFORCE ~3,200 steps/s; A2C 1,950 (n=5) to 2,900 (n=20) steps/s.
    if cfg.algorithm == "reinforce":
        return 3200.0
    fps = 1950.0 if cfg.rollout_steps <= 5 else 2400.0 if cfg.rollout_steps < 20 else 2900.0
    return fps * (0.8 if not cfg.shared_torso else 1.0)


def plan_table():
    rows = []
    for exp_id, exp in EXPERIMENTS.items():
        for tag in exp["runs"]:
            r, status = load_results(tag), run_status(tag)
            row = {"experiment": exp_id, "run": tag, "status": status,
                   "change": EARLIER_RUNS.get(tag) or NEW_RUNS[tag][1],
                   "frames done": f"{r['frames']:,}" if r else "0"}
            if tag in NEW_RUNS and status != "done":
                cfg = NEW_RUNS[tag][0]
                remaining = cfg.max_frames - (r["frames"] if r and status.startswith("partial (resumes") else 0)
                typical = 16_000_000 if cfg.algorithm == "reinforce" else 4_000_000   # frames the references needed
                fps = estimated_fps(cfg)
                row["expected min"] = round(min(typical, remaining) / fps / 60)
                row["max min (cap)"] = round(remaining / fps / 60)
                row["queued"] = bool(RUN_EXPERIMENTS.get(exp_id))
            rows.append(row)
    df = pd.DataFrame(rows)
    display(df)
    # The queued / time columns only exist while some new run is unfinished.
    queued = df["queued"].fillna(False).astype(bool) if "queued" in df else pd.Series(False, index=df.index)
    todo = df[df.run.isin(NEW_RUNS) & (df.status != "done") & queued].drop_duplicates("run")
    if todo.empty:
        print("New runs still to train in enabled experiments: 0")
    else:
        print(f"New runs still to train in enabled experiments: {len(todo)}  |  expected about "
              f"{todo['expected min'].sum() / 60:.1f} h, at most {todo['max min (cap)'].sum() / 60:.1f} h")
    return df


PLAN = plan_table()

## 13. Run the queue of new experiments

Only runs listed in `NEW_RUNS`, in experiments switched on in `RUN_EXPERIMENTS`, are trained, in the order of
`RUN_ORDER`. **Earlier runs are never retrained**, whatever their status. A new run is skipped when it is
already finished, and resumed from `latest.pt` when it was interrupted. If a run folder holds results with
a different configuration, the run is skipped with a warning instead of overwriting them.

To stop cleanly between runs, create `CMPE260_Project2/STOP`. **Kernel → Interrupt** also works: the current
run saves its checkpoint and the queue stops. Re-run the cell to continue.

In [ ]:
RUN_TRAINING = True   # False: skip training and only rebuild the tables below
QUEUE_LOG = P2_ROOT / "final_queue_log.jsonl"


def run_queue():
    if not TRAINING_AVAILABLE:
        print("Analysis-only mode: no GPU, nothing is trained.")
        return
    done_in_session = set()
    for exp_id in RUN_ORDER:
        if not RUN_EXPERIMENTS.get(exp_id):
            continue
        for tag in EXPERIMENTS[exp_id]["runs"]:
            if tag not in NEW_RUNS or tag in done_in_session:
                continue          # earlier runs are never retrained
            done_in_session.add(tag)
            if (P2_ROOT / "STOP").exists():
                print("STOP file found - queue paused. Delete CMPE260_Project2/STOP and re-run this cell.")
                return
            cfg = NEW_RUNS[tag][0]
            prev = load_results(tag)
            if prev is not None and not config_matches(prev["config"], cfg):
                print(f"SKIPPING {tag}: its folder holds results with a different config. Rename one of them.")
                continue
            if prev is not None and finished(prev, cfg):
                print(f"{exp_id} {tag}: already done ({prev['status']}, {prev['frames']:,} frames)")
                continue
            print(f"\n{'=' * 25} {exp_id}: {tag} ({NEW_RUNS[tag][1]}) {'=' * 25}")
            t0 = time.time()
            try:
                res = run_or_resume(tag, cfg)
            except Exception as exc:  # one failed run must not lose the rest of an overnight queue
                import traceback
                traceback.print_exc()
                res = {"status": "error", "error": repr(exc)}
            entry = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "experiment": exp_id, "run": tag,
                     "status": res.get("status"), "frames": res.get("frames"), "gpu": GPU_NAME,
                     "mean_reward_100": res.get("mean_reward_100"), "minutes_this_session": (time.time() - t0) / 60}
            with open(QUEUE_LOG, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(entry) + "\n")
            if res.get("status") == "interrupted":
                print("Run interrupted - queue stopped. Re-run this cell to resume.")
                return
    print("\nQueue finished.")


if RUN_TRAINING:
    run_queue()

## 14. Evaluation of the REINFORCE and A2C models, and critic bias

Every REINFORCE / A2C run's `best.pt` plays 30 seeded games, **sampling from its policy** (stochastic mode,
the policy it was trained as).

**Critic bias.** At every step the A2C critic's estimate V(s) is recorded and compared with the **actual
discounted return** G_t from that step (γ = 0.99): bias = mean(V(s) − G_t). A positive value means
over-estimation. REINFORCE has no value estimate (its gradient uses the actual returns), so it has no
estimate bias.

Results are cached in `CMPE260_Project2/eval_<run>_stochastic.json`. When a run's `best.pt` is not on this
machine (only two checkpoints are in the repository), its cached evaluation is used.

Project 1 (DQN) is **not evaluated or re-run here**. Its recorded results are only read in Section 17.

In [ ]:
def checkpoint_frame(path):
    return int(torch.load(path, map_location="cpu", weights_only=False).get("frame", -1))


def evaluate_cached(tag, mode="stochastic"):
    # REINFORCE / A2C runs only.
    ckpt = P2_ROOT / tag / "best.pt"
    cache = P2_ROOT / f"eval_{tag}_{mode}.json"
    cached = json.loads(cache.read_text(encoding="utf-8")) if cache.exists() else None
    if not ckpt.exists():
        if cached is not None:
            return cached       # no checkpoint on this machine: use the saved evaluation
        print(f"{tag:34s} not evaluated (no best.pt and no saved evaluation)")
        return None
    frame = checkpoint_frame(ckpt)
    if (cached is not None and not RERUN_EVAL and cached.get("checkpoint_frame") == frame
            and cached.get("episodes") == EVAL_EPISODES):
        return cached
    if not TRAINING_AVAILABLE:
        print(f"{tag:34s} not evaluated (analysis-only mode)")
        return cached
    t0 = time.perf_counter()
    res = play_and_measure(pg_policy(load_pg(tag), mode == "stochastic", SEED))
    res.update({"label": tag, "run_tag": tag, "mode": mode, "checkpoint_frame": frame,
                "eval_minutes": (time.perf_counter() - t0) / 60, "device": GPU_NAME})
    atomic_json(res, cache)
    print(f"{tag:34s} score {res['mean']:+.2f} ± {res['std']:.2f}  "
          f"bias {res.get('bias_mean', float('nan')):+.3f}  (best.pt @ {frame:,} frames)")
    return res


ALL_PG_RUNS = list(dict.fromkeys(tag for exp in EXPERIMENTS.values() for tag in exp["runs"]))
EVAL_RESULTS = {}
for tag in ALL_PG_RUNS:
    if load_results(tag) is not None:
        r = evaluate_cached(tag)
        if r is not None:
            EVAL_RESULTS[(tag, "stochastic")] = r
print(f"Evaluations available for {len(EVAL_RESULTS)} REINFORCE / A2C runs.")

## 15. Results

**15.1 All runs.** One row per REINFORCE / A2C run, earlier and new. `frames to +15` and `frames to solve`
(mean of the last 100 games ≥ 19, the Project 1 rule) measure convergence speed; `reward_100 @5M/10M/20M`
compare runs at equal frame budgets (for runs that solved earlier, the final value is carried forward).

`learning-signal std` measures the **variance** side of the bias-variance trade-off: the spread of the weight
that multiplies ∇log π in each batch (the return for REINFORCE, the advantage for A2C and for REINFORCE with a
learned baseline), in reward units, averaged over training. Lower means a less noisy gradient. It is read
from the `train/return_std` and `train/advantage_std` TensorBoard logs.

Saved as `CMPE260_Project2/final_pg_results.csv` and `.md`.

In [ ]:
import logging
logging.getLogger("tensorboard").setLevel(logging.ERROR)   # hide "out of order event" notices from resumed runs


def first_crossing(df, threshold, min_episodes=1):
    ok = (df.reward_100.values >= threshold) & (np.arange(1, len(df) + 1) >= min_episodes)
    hits = np.flatnonzero(ok)
    if not hits.size:
        return None
    i = int(hits[0])
    return {"frame": int(df.frame.iloc[i]), "episodes": i + 1,
            "wall": float(df.wall_minutes.iloc[i]) if "wall_minutes" in df else float("nan")}


def fmt(x, spec="{:,.0f}"):
    return "—" if x is None or (isinstance(x, float) and not np.isfinite(x)) else spec.format(x)


def fmt_int(x):
    return "—" if x is None else f"{int(x):,}"


def reward_at(log, status, frame):
    if log.frame.iloc[-1] < frame:
        return float(log.reward_100.iloc[-1]) if status == "solved" else float("nan")
    return float(log.reward_100.iloc[np.searchsorted(log.frame.values, frame)])


def describe(cfg):
    a2c = cfg["algorithm"] == "a2c"
    parts = [("n=" if a2c else "T=") + str(cfg["rollout_steps"]),
             f"lr {cfg['learning_rate']:g}" + (" linear" if cfg.get("lr_schedule") == "linear" else ""),
             cfg["optimizer"]]
    if cfg["num_envs"] != 16:
        parts.append(f"{cfg['num_envs']} envs")
    if a2c:
        if cfg["gae_lambda"] != 1.0:
            parts.append(f"λ={cfg['gae_lambda']:g}")
        if cfg["value_coef"] != 0.5:
            parts.append(f"vcoef {cfg['value_coef']:g}")
        if not cfg.get("shared_torso", True):
            parts.append("separate critic")
    else:
        if not cfg.get("normalize_returns", True):
            parts.append("no return std.")
        if cfg.get("value_baseline"):
            parts.append("V(s) baseline")
    if cfg["entropy_coef"] != 0.01:
        parts.append(f"ent {cfg['entropy_coef']:g}")
    return ", ".join(parts)


def tb_mean(tag, scalar):
    # (mean over the whole run, mean over the last 20% of its frames) of a logged training scalar
    steps, vals = tensorboard_curve(P2_ROOT / tag / "tensorboard", scalar)
    if not len(vals):
        return None, None
    return float(vals.mean()), float(vals[steps >= 0.8 * steps[-1]].mean())


def run_metrics(tag):
    r, log = load_results(tag), p2_episode_log(tag)
    if r is None or log is None or not len(log):
        return None
    c15, c19 = first_crossing(log, 15.0), first_crossing(log, 19.0, 100)
    ev = EVAL_RESULTS.get((tag, "stochastic"))
    has_value = r["algorithm"] == "a2c" or bool(r["config"].get("value_baseline"))
    signal, signal_late = tb_mean(tag, "train/advantage_std" if has_value else "train/return_std")
    _, ev_late = tb_mean(tag, "train/explained_variance") if has_value else (None, None)
    return {
        "run": tag, "algorithm": r["algorithm"], "settings": describe(r["config"]), "seed": r["config"]["seed"],
        "status": r["status"], "origin": "earlier" if tag in EARLIER_RUNS else "new",
        "frames": r["frames"], "frames_to_15": c15 and c15["frame"], "frames_to_solve": c19 and c19["frame"],
        "episodes_to_solve": c19 and c19["episodes"], "wall_to_solve": c19 and c19["wall"],
        "r5M": reward_at(log, r["status"], 5e6), "r10M": reward_at(log, r["status"], 10e6),
        "r20M": reward_at(log, r["status"], 20e6),
        "best": float(log.reward_100.max()), "final": float(log.reward_100.iloc[-1]),
        "auc10M": normalized_auc(log.frame, log.reward_100, 10_000_000),
        "updates": r["updates"], "samples": r["samples_used"], "wall": r["wall_clock_minutes"], "fps": r["avg_fps"],
        "gpu": r.get("gpu", "?"), "num_envs": r["config"]["num_envs"], "params": r.get("num_params"),
        "eval_mean": ev and ev["mean"], "eval_std": ev and ev["std"],
        "bias": ev.get("bias_mean") if ev else None, "abs_err": ev.get("abs_error_mean") if ev else None,
        "over_frac": ev.get("overestimated_fraction") if ev else None,
        "signal_std": signal, "signal_std_late": signal_late, "explained_var_late": ev_late,
        "log": log, "results": r,
    }


METRICS = {tag: m for tag in ALL_PG_RUNS if (m := run_metrics(tag)) is not None}


def display_row(m):
    return {
        "run": m["run"], "algorithm": m["algorithm"], "settings": m["settings"], "seed": m["seed"],
        "origin": m["origin"], "status": m["status"], "frames": fmt(m["frames"]),
        "frames to +15": fmt(m["frames_to_15"]), "frames to solve": fmt(m["frames_to_solve"]),
        "episodes to solve": fmt(m["episodes_to_solve"]), "wall-min to solve": fmt(m["wall_to_solve"], "{:.1f}"),
        "reward_100 @5M": fmt(m["r5M"], "{:+.1f}"), "@10M": fmt(m["r10M"], "{:+.1f}"), "@20M": fmt(m["r20M"], "{:+.1f}"),
        "best reward_100": fmt(m["best"], "{:+.2f}"), "final reward_100": fmt(m["final"], "{:+.2f}"),
        "AUC@10M": fmt(m["auc10M"], "{:.3f}"), "updates": fmt(m["updates"]),
        "wall-clock min": fmt(m["wall"], "{:.1f}"), "steps/s": fmt(m["fps"]),
        "eval (stochastic)": "—" if m["eval_mean"] is None else f"{m['eval_mean']:+.2f} ± {m['eval_std']:.2f}",
        "learning-signal std": fmt(m["signal_std"], "{:.3f}"),
        "V(s) bias": fmt(m["bias"], "{:+.3f}"), "GPU": m["gpu"],
    }


ALL_RESULTS = pd.DataFrame([display_row(m) for m in METRICS.values()]).set_index("run")
ALL_RESULTS.to_csv(P2_ROOT / "final_pg_results.csv", encoding="utf-8-sig")
(P2_ROOT / "final_pg_results.md").write_text(ALL_RESULTS.to_markdown(), encoding="utf-8")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
display(ALL_RESULTS)
missing = [t for t in ALL_PG_RUNS if t not in METRICS]
print(f"{len(METRICS)} runs with results; {len(missing)} not run yet: {', '.join(missing) if missing else 'none'}")
gpus = sorted({m["gpu"] for m in METRICS.values()})
if len(gpus) > 1:
    print("NOTE: these runs used different GPUs:", "; ".join(gpus),
          "- compare wall-clock minutes only between runs on the same GPU (the GPU column shows which).")

**15.2 One table per experiment.** The first row of each table is the experiment's reference run. `Δ frames to
solve` is the change relative to the reference (negative = solved faster). The `GPU` column shows where each
run was trained: frames are comparable across GPUs, wall-clock minutes only between runs on the same GPU.
All tables are also saved together in `CMPE260_Project2/final_experiment_tables.md`, ready for the report.

In [ ]:
EXP_COLUMNS = ["role", "change", "status", "frames to +15", "frames to solve", "Δ frames to solve",
               "wall-min to solve", "best reward_100", "final reward_100", "eval (stochastic)",
               "learning-signal std", "V(s) bias", "GPU"]


def run_finished(m):
    cfg = PGConfig(**{k: v for k, v in m["results"]["config"].items() if k in PGConfig.__dataclass_fields__})
    return finished(m["results"], cfg)


def experiment_table(exp_id):
    exp = EXPERIMENTS[exp_id]
    ref = METRICS.get(exp["runs"][0])
    rows = []
    for i, tag in enumerate(exp["runs"]):
        m = METRICS.get(tag)
        row = {"run": tag, "role": "reference" if i == 0 else ("earlier" if tag in EARLIER_RUNS else "new"),
               "change": "—" if i == 0 else (NEW_RUNS[tag][1] if tag in NEW_RUNS else EARLIER_RUNS[tag]),
               "status": run_status(tag)}
        if m is None:
            rows.append({**{c: "—" for c in EXP_COLUMNS}, **row})
            continue
        d = display_row(m)
        delta = "—"
        if i > 0 and ref and ref["frames_to_solve"] and m["frames_to_solve"]:
            delta = f"{100 * (m['frames_to_solve'] / ref['frames_to_solve'] - 1):+.0f}%"
        elif i > 0 and m["status"] != "solved" and run_finished(m):
            delta = "not solved"
        rows.append({**{c: d[c] for c in EXP_COLUMNS if c in d}, **row, "Δ frames to solve": delta,
                     "status": m["status"] if tag in EARLIER_RUNS else row["status"]})
    return pd.DataFrame(rows).set_index("run")[EXP_COLUMNS]


md_parts = []
for exp_id, exp in EXPERIMENTS.items():
    table = experiment_table(exp_id)
    print(f"\n{exp_id}: {exp['title']}")
    display(table)
    md_parts.append(f"### {exp_id}: {exp['title']}\n\n{table.to_markdown()}\n")
(P2_ROOT / "final_experiment_tables.md").write_text("\n".join(md_parts), encoding="utf-8")
print("Saved:", P2_ROOT / "final_experiment_tables.md")

**15.3 Seed summaries.** Mean ± standard deviation over seeds 42–44 for the configurations that were run with
three seeds. Use these numbers for the main claims of the report; single-seed differences smaller than the
seed-to-seed spread shown here should be treated as inconclusive.

In [ ]:
SEED_GROUPS = {
    "REINFORCE, T=256 (R1)": ["reinforce_T256_seed42", "reinforce_T256_seed43", "reinforce_T256_seed44"],
    "A2C, n=5 (E2)":         ["a2c_seed42", "a2c_seed43", "a2c_seed44"],
    "A2C, n=20 (A1)":        ["a2c_n20_seed42", "a2c_n20_seed43", "a2c_n20_seed44"],
}


def mean_std(values, spec="{:,.0f}"):
    vals = [float(v) for v in values if v is not None and np.isfinite(float(v))]
    if not vals:
        return "—"
    if len(vals) == 1:
        return spec.format(vals[0]) + " (1 seed)"
    std_spec = spec.replace("+", "")   # no sign on the spread
    return f"{spec.format(np.mean(vals))} ± {std_spec.format(np.std(vals, ddof=1))} (n={len(vals)})"


seed_rows = []
for name, tags in SEED_GROUPS.items():
    ms = [METRICS[t] for t in tags if t in METRICS]
    seed_rows.append({
        "configuration": name, "seeds with results": f"{len(ms)} of {len(tags)}",
        "solved": f"{sum(m['status'] == 'solved' for m in ms)} of {len(ms)}",
        "frames to +15": mean_std([m["frames_to_15"] for m in ms]),
        "frames to solve": mean_std([m["frames_to_solve"] for m in ms]),
        "wall-min to solve": mean_std([m["wall_to_solve"] for m in ms], "{:.1f}"),
        "eval (stochastic)": mean_std([m["eval_mean"] for m in ms], "{:+.2f}"),
        "learning-signal std": mean_std([m["signal_std"] for m in ms], "{:.3f}"),
        "V(s) bias": mean_std([m["bias"] for m in ms], "{:+.3f}"),
    })
SEED_SUMMARY = pd.DataFrame(seed_rows).set_index("configuration")
display(SEED_SUMMARY)

for prefix in ("reinforce", "a2c"):
    path = P2_ROOT / f"sweep_{prefix}.csv"
    if path.exists():
        print(f"\nFirst-run hyper-parameter sweep ({prefix}, {'3M' if prefix == 'reinforce' else '2M'} frames per trial):")
        display(pd.read_csv(path))

## 16. Learning curves of the experiments

* One panel per experiment: mean reward of the last 100 games against agent steps (dotted line = +15,
  dashed line = solved). Saved as `final_experiment_curves.png`.
* **A2C n-step trade-off** (experiment A2): frames to solve, critic bias and advantage spread (variance)
  against n. Saved as `a2c_nstep_tradeoff.png`.

In [ ]:
exp_ids = list(EXPERIMENTS)
ncols = 4
nrows = math.ceil(len(exp_ids) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.0 * nrows), squeeze=False)
for ax, exp_id in zip(axes.flat, exp_ids):
    for i, tag in enumerate(EXPERIMENTS[exp_id]["runs"]):
        if tag in METRICS:
            log = METRICS[tag]["log"]
            label = ("ref: " if i == 0 else "") + (NEW_RUNS[tag][1] if tag in NEW_RUNS else tag)
            ax.plot(log.frame / 1e6, log.reward_100, lw=2.2 if i == 0 else 1.3, label=label)
    ax.axhline(15, ls=":", c="gray", lw=1)
    ax.axhline(19, ls="--", c="gray", lw=1)
    ax.set_title(f"{exp_id}: {EXPERIMENTS[exp_id]['title']}", fontsize=9)
    ax.set_xlabel("agent steps (millions)", fontsize=8)
    ax.set_ylabel("mean reward (last 100)", fontsize=8)
    ax.set_ylim(-21.5, 21.5)
    ax.grid(alpha=0.3)
    if ax.lines:
        ax.legend(fontsize=7, loc="lower right")
for ax in list(axes.flat)[len(exp_ids):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(P2_ROOT / "final_experiment_curves.png", dpi=130)
plt.show()

# A2C n-step trade-off (experiment A2)
nstep = sorted(((METRICS[t]["results"]["config"]["rollout_steps"], METRICS[t]) for t in EXPERIMENTS["A2"]["runs"]
                if t in METRICS), key=lambda x: x[0])
if len(nstep) >= 2:
    fig, ax1 = plt.subplots(figsize=(7.5, 4.4))
    ns = [n for n, _ in nstep]
    ax1.plot(ns, [m["frames_to_solve"] / 1e6 if m["frames_to_solve"] else np.nan for _, m in nstep], "o-",
             label="frames to solve (M)")
    ax1.set_xscale("log", base=2)
    ax1.set_xticks(ns, [str(n) for n in ns])
    ax1.set_xlabel("n (steps before bootstrapping)")
    ax1.set_ylabel("frames to solve (millions)")
    ax2 = ax1.twinx()
    nan = lambda v: np.nan if v is None else v
    ax2.plot(ns, [nan(m["bias"]) for _, m in nstep], "s--", c="tab:red", label="critic bias V − G")
    ax2.plot(ns, [nan(m["abs_err"]) for _, m in nstep], "^:", c="tab:purple", label="mean |V − G|")
    ax2.plot(ns, [nan(m["signal_std"]) for _, m in nstep], "d-.", c="tab:green", label="advantage std (variance)")
    ax2.axhline(0, c="gray", lw=0.8)
    ax2.set_ylabel("reward units")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper center")
    ax1.set_title("A2C: n-step length vs speed, bias and variance")
    ax1.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(P2_ROOT / "a2c_nstep_tradeoff.png", dpi=150)
    plt.show()

## 17. Final tuned hyper-parameters, and the comparison with DQN (Project 1)

### 17.1 Final tuned hyper-parameters

The **best configuration of each algorithm** is picked automatically from all REINFORCE / A2C runs, with a
rule that favours results confirmed over several seeds:

1. configurations whose runs **all solved** Pong come first;
2. then configurations with **more seeds** (up to 3) — a single lucky run cannot displace a 3-seed result;
3. then the **lowest mean frames to solve** (hardware-independent).

Single-seed configurations that solved faster than the selected one are listed as *candidates to confirm with
more seeds*. To choose by hand instead, set `BEST_OVERRIDE`. The table lists every setting of the selected
configurations, next to the Project 1 DQN settings (read from its recorded `results.json`).

In [ ]:
SELECTION_IGNORED = IGNORED_FIELDS | {"seed"}
BEST_OVERRIDE = {"reinforce": None, "a2c": None}   # e.g. {"a2c": "a2c_n40_seed42"} to choose by hand


def full_config(cfg):
    return {**asdict(PGConfig(algorithm=cfg["algorithm"])), **cfg}


def config_key(cfg):
    return tuple(sorted((k, v) for k, v in full_config(cfg).items() if k not in SELECTION_IGNORED))


def configuration_groups(algorithm):
    groups = {}
    for m in METRICS.values():
        if m["algorithm"] == algorithm:
            groups.setdefault(config_key(m["results"]["config"]), []).append(m)
    return [sorted(g, key=lambda m: m["seed"]) for g in groups.values()]


def mean_frames_to_solve(ms):
    solved = [m["frames_to_solve"] for m in ms if m["frames_to_solve"]]
    return float(np.mean(solved)) if solved else float("inf")


def selection_rank(ms):
    all_solved = all(m["frames_to_solve"] for m in ms)
    return (not all_solved, -min(len(ms), 3), mean_frames_to_solve(ms))


BEST = {}
for algo in ("reinforce", "a2c"):
    groups = configuration_groups(algo)
    if not groups:
        continue
    if BEST_OVERRIDE.get(algo):
        BEST[algo] = next(g for g in groups if any(m["run"] == BEST_OVERRIDE[algo] for m in g))
    else:
        BEST[algo] = min(groups, key=selection_rank)
    best = BEST[algo]
    print(f"Best {algo}: {best[0]['settings']}  ({len(best)} seed(s): {', '.join(m['run'] for m in best)}; "
          f"mean frames to solve {mean_frames_to_solve(best):,.0f})")
    for g in groups:
        if g is not best and len(g) < len(best) and all(m["frames_to_solve"] for m in g) \
                and mean_frames_to_solve(g) < mean_frames_to_solve(best):
            print(f"   candidate to confirm with more seeds: {g[0]['settings']} "
                  f"({', '.join(m['run'] for m in g)}: {mean_frames_to_solve(g):,.0f} frames)")

TUNING_EVIDENCE = {
    "reinforce": "first sweep (lr × rollout, 3M frames per trial); E1 (20M-frame variants); R1–R4",
    "a2c": "first sweep (lr, optimiser, n, λ; 2M frames per trial); E2, E3; A1–A8",
}


def pg_hyperparameters(algo, ms):
    c = full_config(ms[0]["results"]["config"])
    a2c = algo == "a2c"
    T, N = c["rollout_steps"], c["num_envs"]
    if a2c:
        target = f"{T}-step bootstrapped return" + (f", GAE λ = {c['gae_lambda']:g}" if c["gae_lambda"] < 1 else " (λ = 1)")
    else:
        target = ("Monte-Carlo return, restarted at every point" if c["point_reset"] else "Monte-Carlo return per game") + \
                 (", standardised per batch" if c["normalize_returns"] else "") + \
                 (", learned V(s) baseline" if c.get("value_baseline") else "")
    return {
        "selected run(s)": ", ".join(m["run"] for m in ms),
        "optimiser": "Adam (ε = 1e-5)" if c["optimizer"] == "adam" else f"RMSprop (α = {c['rmsprop_alpha']:g}, ε = {c['rmsprop_eps']:g})",
        "learning rate": f"{c['learning_rate']:g}, " + ("linear decay to 0" if c["lr_schedule"] == "linear" else "constant"),
        "discount γ": f"{c['gamma']:g}",
        "parallel environments": str(N),
        "data per gradient update": f"{T} steps × {N} envs = {T * N:,} states" + ("" if a2c else " (only completed points)"),
        "learning target": target,
        "entropy coefficient": f"{c['entropy_coef']:g}",
        "value-loss coefficient": f"{c['value_coef']:g}" if (a2c or c.get("value_baseline")) else "— (no critic)",
        "gradient-norm clipping": f"{c['max_grad_norm']:g}",
        "exploration": "sampling from π, plus the entropy bonus",
        "replay buffer / target network": "none (on-policy)",
        "network": ("shared CNN torso with policy and value heads" if c.get("shared_torso", True)
                    else "separate CNNs for actor and critic") if a2c else "CNN torso with a policy head",
        "how it was tuned": TUNING_EVIDENCE[algo],
    }


def dqn_hyperparameters(tag):
    # Read from the recorded Project 1 results (nothing from Project 1 is run here).
    path = P1_ROOT / tag / "results.json"
    if not path.exists():
        return None
    c = json.loads(path.read_text(encoding="utf-8"))["config"]
    return {
        "selected run(s)": f"{tag} (Project 1 best)",
        "optimiser": "Adam (PyTorch defaults)",
        "learning rate": f"{c['learning_rate']:g}, constant",
        "discount γ": f"{c['gamma']:g}",
        "parallel environments": "1",
        "data per gradient update": f"batch of {c['batch_size']} transitions from replay, one update per step "
                                    f"after {c['warmup_steps']:,} warm-up steps",
        "learning target": "1-step TD target r + γ · max_a′ Q_target(s′, a′)",
        "entropy coefficient": "—",
        "value-loss coefficient": "— (MSE TD loss)",
        "gradient-norm clipping": "none",
        "exploration": f"NoisyNet (σ₀ = {c['noisy_sigma0']:g})",
        "replay buffer / target network": f"prioritized replay ({c['replay_capacity']:,} transitions, α = {c['per_alpha']:g}, "
                                          f"β {c['per_beta_start']:g} → 1 over {c['per_beta_frames']:,} frames); "
                                          f"target network synced every {c['target_sync_steps']:,} steps",
        "network": "CNN torso with noisy fully connected layers (Q head)",
        "how it was tuned": "Project 1: ε-greedy vs Boltzmann vs NoisyNet exploration, then PER on the best",
    }


hp = {}
if (d := dqn_hyperparameters("per_noisynet_seed42")) is not None:
    hp["DQN NoisyNet + PER (Project 1)"] = d
for algo, name in (("reinforce", "REINFORCE (best)"), ("a2c", "A2C (best)")):
    if algo in BEST:
        hp[name] = pg_hyperparameters(algo, BEST[algo])
FINAL_HYPERPARAMETERS = pd.DataFrame(hp)
FINAL_HYPERPARAMETERS.to_csv(P2_ROOT / "final_hyperparameters.csv", encoding="utf-8-sig")
(P2_ROOT / "final_hyperparameters.md").write_text(FINAL_HYPERPARAMETERS.to_markdown(), encoding="utf-8")
display(FINAL_HYPERPARAMETERS)
print("Saved:", P2_ROOT / "final_hyperparameters.md")

### 17.2 Final comparison: DQN vs REINFORCE vs A2C

**Project 1 is only read, never re-run.** The DQN columns use the recorded Project 1 results in
`CMPE260_Project1/<run>/`: `results.json` and the TensorBoard logs for training, and `evaluation_bias_*.json`
for the evaluation score and the Q-value over-estimation (Q(s,a) − actual discounted return, measured on the
Project 1 models with greedy play, because Q-learning estimates the value of the greedy policy). The
REINFORCE and A2C columns use the first tuned configuration and the best configuration from 17.1; for
configurations run with several seeds the table shows mean ± std.

> **Hardware differs between the projects.** DQN (Project 1) ran on Google Colab (NVIDIA L4 GPU)
> with **one** environment and a gradient update on every step. REINFORCE and A2C (Project 2) ran on NVIDIA
> RTX GPUs with **16 parallel environments** (the exact GPU of each run is in the `hardware` row). Wall-clock
> minutes and steps per second (marked †) are therefore **not directly comparable** across projects: most of
> the gap comes from parallel environment sampling, not from GPU speed. Frames, episodes, gradient updates,
> samples, evaluation scores, bias and variance are hardware-independent and are the basis of the comparison.

Saved as `CMPE260_Project2/final_comparison.csv` and `.md`, and the plots as `final_comparison_curves.png`
and `final_value_bias.png`.

In [ ]:
P1_RUNS = {
    "baseline_epsilon_seed42":            "DQN ε-greedy baseline (P1)",
    "noisynet_seed42":                    "DQN NoisyNet (P1)",
    "per_noisynet_seed42":                "DQN NoisyNet + PER (P1 best)",
    "boltzmann_softmax_decay100k_seed42": "DQN Boltzmann, tuned (P1)",
}


def p1_eval(tag, mode):
    path = P1_ROOT / tag / f"evaluation_bias_{mode}.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


def p1_episode_log(run_tag, results):
    steps, rewards = tensorboard_curve(P1_ROOT / run_tag / "tensorboard", "episode/reward")
    if not len(steps):
        return None
    df = pd.DataFrame({"frame": steps, "reward": rewards})
    df["reward_100"] = df.reward.rolling(100, min_periods=1).mean()
    w_steps, w_vals = tensorboard_curve(P1_ROOT / run_tag / "tensorboard", "system/wall_minutes")
    if len(w_steps) > 1:
        df["wall_minutes"] = np.interp(df.frame, w_steps, w_vals)
    else:
        df["wall_minutes"] = df.frame / max(df.frame.iloc[-1], 1) * results.get("wall_clock_minutes", np.nan)
    return df


def dqn_metrics(tag):
    rpath = P1_ROOT / tag / "results.json"
    if not rpath.exists():
        return None
    r = json.loads(rpath.read_text(encoding="utf-8"))
    log = p1_episode_log(tag, r)
    if log is None:
        return None
    cfg = r.get("config", {})
    frames = int(r["frames"])
    updates = max(frames - int(cfg.get("warmup_steps", 10_000)), 0)     # one update per step after warm-up
    c15, c19 = first_crossing(log, 15.0), first_crossing(log, 19.0, 100)
    return {"status": r["status"], "frames": frames, "frames_to_15": c15 and c15["frame"],
            "frames_to_solve": c19 and c19["frame"], "episodes_to_solve": c19 and c19["episodes"],
            "wall_to_solve": c19 and c19["wall"], "wall": r["wall_clock_minutes"],
            "fps": frames / (r["wall_clock_minutes"] * 60), "best": float(log.reward_100.max()),
            "final": float(log.reward_100.iloc[-1]), "updates": updates,
            "samples": updates * int(cfg.get("batch_size", 32)), "gpu": r.get("gpu", "?"),
            "eval_greedy": p1_eval(tag, "greedy"), "eval_eps": p1_eval(tag, "eps0.02"), "log": log}


# Recorded Project 1 evaluations: score and Q-value over-estimation of each DQN model
dqn_rows = []
for tag, label in P1_RUNS.items():
    g, e = p1_eval(tag, "greedy"), p1_eval(tag, "eps0.02")
    if g or e:
        dqn_rows.append({
            "model": label, "checkpoint frame": fmt_int((g or e).get("checkpoint_frame")),
            "score greedy": "—" if not g else f"{g['mean']:+.2f} ± {g['std']:.2f}",
            "score ε=0.02": "—" if not e else f"{e['mean']:+.2f} ± {e['std']:.2f}",
            "mean Q estimate (greedy)": "—" if not g else f"{g['estimate_mean']:.3f}",
            "mean actual return (greedy)": "—" if not g else f"{g['mc_return_mean']:.3f}",
            "Q bias greedy": "—" if not g else f"{g['bias_mean']:+.3f}",
            "Q bias ε=0.02": "—" if not e else f"{e['bias_mean']:+.3f}",
            "% steps over-estimated (greedy)": "—" if not g else f"{100 * g['overestimated_fraction']:.0f}%",
        })
if dqn_rows:
    print("Project 1 DQN models (recorded results): score and Q-value over-estimation")
    display(pd.DataFrame(dqn_rows).set_index("model"))

FAMILY = {
    "dqn": ("value-based (learns Q(s,a))", "off-policy: replay buffer, each frame reused", "1-step TD with max over actions", "Q(s,a)"),
    "reinforce": ("policy gradient (learns π(a|s))", "on-policy: each sample used once", "Monte-Carlo return (per point)", "none (actual returns)"),
    "a2c": ("actor-critic (learns π(a|s) and V(s))", "on-policy: each sample used once", "n-step bootstrapped return", "V(s) critic"),
}


def short_name(algo, ms):
    c = ms[0]["results"]["config"]
    return ("REINFORCE, " if algo == "reinforce" else "A2C, ") + describe(c)


# Columns of the final table: both DQN results, then the first tuned and the best configuration of each
# policy-gradient method (the best one is selected in 17.1; it is shown once if it equals the first tuned).
COMPARISON_METHODS = [("DQN ε-greedy baseline (P1)", "dqn", ["baseline_epsilon_seed42"]),
                      ("DQN NoisyNet + PER (P1 best)", "dqn", ["per_noisynet_seed42"])]
FIRST_TUNED = {"reinforce": ["reinforce_seed42"], "a2c": SEED_GROUPS["A2C, n=5 (E2)"]}
for algo in ("reinforce", "a2c"):
    first = [t for t in FIRST_TUNED[algo] if t in METRICS]
    best = [m["run"] for m in BEST.get(algo, [])]
    if first and set(first) != set(best):
        COMPARISON_METHODS.append((short_name(algo, [METRICS[first[0]]]) + " (first tuned)", algo, first))
    if best:
        label = short_name(algo, BEST[algo]) + (" (best = first tuned)" if set(first) == set(best) else " (best)")
        COMPARISON_METHODS.append((label, algo, best))


def comparison_column(kind, tags):
    family, data, target, estimator = FAMILY[kind]
    col = {"family": family, "data use": data, "learning target": target, "value estimator": estimator}
    if kind == "dqn":
        ms = [m for t in tags if (m := dqn_metrics(t)) is not None]
        envs = "1"
    else:
        ms = [METRICS[t] for t in tags if t in METRICS]
        envs = ", ".join(sorted({str(m["num_envs"]) for m in ms})) or "16"
    if not ms:
        return {**col, "runs (seeds) with results": f"0 of {len(tags)}"}
    gpus = sorted({m["gpu"] for m in ms})
    col.update({
        "hardware (GPU, parallel envs)": f"{' / '.join(gpus)}; {envs} env" + ("s" if envs != "1" else ""),
        "runs (seeds) with results": f"{len(ms)} of {len(tags)}",
        "solved (mean-100 ≥ 19)": f"{sum(m['status'] == 'solved' for m in ms)} of {len(ms)}",
        "frames to +15": mean_std([m["frames_to_15"] for m in ms]),
        "frames to solve": mean_std([m["frames_to_solve"] for m in ms]),
        "episodes to solve": mean_std([m["episodes_to_solve"] for m in ms]),
        "wall-clock min to solve †": mean_std([m["wall_to_solve"] for m in ms], "{:.1f}"),
        "agent steps / s †": mean_std([m["fps"] for m in ms]),
        "gradient updates (total)": ("≈" if kind == "dqn" else "") + mean_std([m["updates"] for m in ms]),
        "states per gradient update": "32" if kind == "dqn" else mean_std([m["samples"] / max(m["updates"], 1) for m in ms]),
        "sample reuse (gradient samples / frame)": ("≈" if kind == "dqn" else "") + mean_std(
            [m["samples"] / max(m["frames"], 1) for m in ms], "{:.2f}"),
        "best reward_100": mean_std([m["best"] for m in ms], "{:+.2f}"),
    })
    if kind == "dqn":
        g = [m["eval_greedy"] for m in ms if m["eval_greedy"]]
        e = [m["eval_eps"] for m in ms if m["eval_eps"]]
        col.update({
            "evaluation score (30 games)": (f"greedy {g[0]['mean']:+.2f} ± {g[0]['std']:.2f}; " if g else "") +
                                           (f"ε=0.02 {e[0]['mean']:+.2f} ± {e[0]['std']:.2f}" if e else ""),
            "bias: value estimate − actual return": f"{g[0]['bias_mean']:+.3f}" if g else "—",
            "mean |value estimate − actual return|": f"{g[0]['abs_error_mean']:.3f}" if g else "—",
            "% steps over-estimated": f"{100 * g[0]['overestimated_fraction']:.0f}%" if g else "—",
            "variance: learning-signal std (mean over training)": "n/a (value-based; no policy-gradient weight)",
            "variance: learning-signal std (last 20% of training)": "n/a",
            "critic explained variance (last 20% of training)": "n/a",
            "parameters": f"{count_params(NoisyDQN((4, 84, 84), 6) if 'noisynet' in tags[0] else DQN((4, 84, 84), 6)):,}",
        })
    else:
        reinforce = kind == "reinforce"
        col.update({
            "evaluation score (30 games)": "sampled policy " + mean_std([m["eval_mean"] for m in ms], "{:+.2f}"),
            "bias: value estimate − actual return": "n/a (no value estimate; uses actual returns, unbiased)"
                if reinforce else mean_std([m["bias"] for m in ms], "{:+.3f}"),
            "mean |value estimate − actual return|": "n/a" if reinforce else mean_std([m["abs_err"] for m in ms], "{:.3f}"),
            "% steps over-estimated": "n/a" if reinforce else mean_std(
                [100 * m["over_frac"] if m["over_frac"] is not None else None for m in ms], "{:.0f}%"),
            "variance: learning-signal std (mean over training)": mean_std([m["signal_std"] for m in ms], "{:.3f}"),
            "variance: learning-signal std (last 20% of training)": mean_std([m["signal_std_late"] for m in ms], "{:.3f}"),
            "critic explained variance (last 20% of training)": "n/a (no critic)" if reinforce
                else mean_std([m["explained_var_late"] for m in ms], "{:.2f}"),
            "parameters": f"{int(ms[0]['params']):,}" if ms[0]["params"] else "—",
        })
    return col


FINAL_COMPARISON = pd.DataFrame({name: comparison_column(kind, tags) for name, kind, tags in COMPARISON_METHODS})
FINAL_COMPARISON.to_csv(P2_ROOT / "final_comparison.csv", encoding="utf-8-sig")
# The note is built from the GPUs recorded in each run's results, so it stays correct when new runs are added.
_p1_gpus = sorted({json.loads((P1_ROOT / t / "results.json").read_text(encoding="utf-8")).get("gpu", "?")
                   for _, k, tags in COMPARISON_METHODS if k == "dqn" for t in tags if (P1_ROOT / t / "results.json").exists()})
_p2_gpus = sorted({m["gpu"] for m in METRICS.values()})
HARDWARE_NOTE = (
    f"† Hardware differs: DQN (Project 1) was trained on Google Colab ({' / '.join(_p1_gpus)}) with one "
    f"environment; REINFORCE and A2C (Project 2) on {' / '.join(_p2_gpus)} with 16 parallel environments. "
    "Wall-clock times and steps/s are not directly comparable across projects; most of the gap comes from "
    "parallel environment sampling rather than GPU speed. All other rows are hardware-independent.")
(P2_ROOT / "final_comparison.md").write_text(FINAL_COMPARISON.to_markdown() + "\n\n" + HARDWARE_NOTE + "\n",
                                             encoding="utf-8")
display(FINAL_COMPARISON)
print(HARDWARE_NOTE)
print("\nSaved:", P2_ROOT / "final_comparison.csv", "and .md")

In [ ]:
CURVE_RUNS = [(name, kind, tags[0]) for name, kind, tags in COMPARISON_METHODS]
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for label, kind, tag in CURVE_RUNS:
    m = dqn_metrics(tag) if kind == "dqn" else METRICS.get(tag)
    if not m:
        continue
    log = m["log"]
    style = "--" if kind == "dqn" else "-"
    for ax, x in zip(axes, [log.frame / 1e6, log.frame / 1e6, log.wall_minutes]):
        ax.plot(x, log.reward_100, style, lw=1.7, label=f"{label} [{tag}]")
for ax, xl, title in zip(axes, ["agent steps (millions)", "agent steps (millions, log scale)", "wall-clock minutes †"],
                         ["Sample efficiency", "Sample efficiency (log scale)", "Wall-clock (different hardware †)"]):
    ax.axhline(15, ls=":", c="gray", lw=1)
    ax.axhline(19, ls="--", c="gray", lw=1)
    ax.set_xlabel(xl)
    ax.set_ylabel("mean reward (last 100 games)")
    ax.set_ylim(-21.5, 21.5)
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[1].set_xscale("log")
axes[0].legend(loc="lower right", fontsize=7)
fig.text(0.5, -0.02, HARDWARE_NOTE, ha="center", fontsize=8, wrap=True)
plt.tight_layout()
plt.savefig(P2_ROOT / "final_comparison_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Value estimate vs actual discounted return: DQN (recorded Project 1 evaluation) vs the A2C critic
bias_sets = [(P1_RUNS[t], p1_eval(t, "greedy")) for t in ("baseline_epsilon_seed42", "per_noisynet_seed42")]
bias_sets += [(f"A2C {m['run']}", EVAL_RESULTS.get((m["run"], "stochastic")))
              for m in [METRICS[t] for t in ("a2c_seed42",) if t in METRICS] + BEST.get("a2c", [])[:1]]
bias_sets = [(k, v) for k, v in dict(bias_sets).items() if v and "sample_estimates" in v]
if bias_sets:
    fig, axes = plt.subplots(1, len(bias_sets), figsize=(4.6 * len(bias_sets), 4.4), squeeze=False)
    lo = min(min(v["sample_returns"] + v["sample_estimates"]) for _, v in bias_sets) - 0.1
    hi = max(max(v["sample_returns"] + v["sample_estimates"]) for _, v in bias_sets) + 0.1
    for ax, (label, res) in zip(axes[0], bias_sets):
        ax.scatter(res["sample_returns"], res["sample_estimates"], s=3, alpha=0.3)
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_xlabel("actual discounted return G_t")
        ax.set_ylabel("estimate (Q or V)")
        ax.set_title(f"{label}\nbias {res['bias_mean']:+.3f}, corr {res['corr']:.2f}", fontsize=9)
        ax.grid(alpha=0.3)
    plt.suptitle("Points above the diagonal are over-estimates", fontsize=10)
    plt.tight_layout()
    plt.savefig(P2_ROOT / "final_value_bias.png", dpi=150)
    plt.show()

## 18. How to read the results

Write the discussion from the measured tables above. The points below say what to look for and why each
difference happens.

* **Sample efficiency (frames to solve).** DQN is **off-policy**: its replay buffer reuses every frame about
  32 times (`sample reuse`), so it needs the fewest frames. REINFORCE and A2C are **on-policy**: the policy
  gradient is only valid for data from the current policy, so each sample is used once and discarded.
* **REINFORCE vs A2C.** Both are on-policy; the difference is the learning signal. REINFORCE weights each
  action by the actual (Monte-Carlo) return: unbiased, but high-variance, and it must wait for the point to
  end. A2C subtracts a learned baseline V(s) (the advantage) and bootstraps after n steps, which lowers the
  variance and allows far more updates (`gradient updates`), at the cost of some bias from the critic.
* **Bias / over-estimation and variance.** DQN's target takes a **max** over noisy Q estimates, so it
  systematically picks over-estimated actions: expect a clearly positive bias on almost every step. A2C's
  critic is also bootstrapped but has no max, so its bias stays near zero. REINFORCE has no estimate and no
  bias; it pays in variance instead, which the `learning-signal std` rows show: its returns spread far more
  than A2C's advantages. Within A2C, longer n-step returns (A2) and higher GAE λ (A3) are expected to trade
  a little more variance for less critic bias; check the A2 and A3 tables and the n-step plot to see whether
  the runs confirm it.
* **Wall-clock time.** On-policy methods parallelise naturally across environments (16 games per forward pass),
  which is why they run at thousands of steps per second. Project 1's DQN stepped one game and made one update
  per step. **The hardware was different (see the note in Section 17)**, so compare wall-clock only with
  that caveat.
* **Final settings.** Section 17.1 lists the selected hyper-parameters of each algorithm next to DQN's and
  the experiments that justify them.
* **Hyper-parameter sensitivity.** Use the experiment tables: REINFORCE's learning rate (R3, E1), rollout
  length (R2, E1) and the standardisation ablation (R4); A2C's n-step length (A2), GAE λ (A3), entropy (A4,
  where 0 collapses), learning rate (A5), optimiser (A6), value coefficient (A7) and network sharing (A8).
  Compare each difference with the seed spread of Section 15.3 before calling it real.

**Limitations to state in the report.** Except for the three-seed groups, every configuration uses one seed.
DQN and the policy-gradient methods ran on different hardware. REINFORCE's first sweep (3M frames per trial)
was too short to separate its settings, which is why the longer variant runs were added.

### References
1. Williams (1992), *Simple statistical gradient-following algorithms for connectionist RL* (REINFORCE).
2. Mnih et al. (2016), *Asynchronous Methods for Deep Reinforcement Learning* (A3C/A2C).
3. Schulman et al. (2016), *High-Dimensional Continuous Control Using Generalized Advantage Estimation*.
4. Mnih et al. (2015), *Human-level control through deep reinforcement learning* (DQN).
5. Fortunato et al. (2018), *Noisy Networks for Exploration*; Schaul et al. (2016), *Prioritized Experience Replay*.
6. Karpathy (2016), *Deep Reinforcement Learning: Pong from Pixels*.